# RSNA Knee Abnormalities Detection — reading the report, then the knee

**Twelve findings. Four thousand four hundred and seven studies. Fifty-eight of them labelled.**

That ratio is the competition. Fifty-eight expert labels cannot fit twelve heads, but 4,349
studies arrive with the radiologist's own report attached — and the test set does not. So the
report is not an input. It is the *training target*, and the model that ships is imaging-only.

This notebook is the distillation of ~60 notebooks in this repository into one file that runs
end to end. It trains from scratch on a bare Kaggle box (DINOv2 weights attached, nothing else)
and writes a real `submission.csv`. If a fine-tuned weights package *is* attached it skips
training and infers from it instead. Both paths write the same file.

## What the corpus established, in the order it was learned

| step | public LB | what changed |
|---|---|---|
| chance | 0.500 | — |
| ImageNet features, frozen | ~0.47 | frozen features do not transfer to knee MRI |
| DINOv2-S fine-tuned @224 | 0.866 | **fine-tuning is worth +0.09; resolution +0.017** |
| same, @336 | 0.883 | 1 mm tear needs ≤0.5 mm/px; 130 mm ÷ 336 = 0.39 mm |
| 20 × DINOv2-S, 5 folds | 0.891 | rank-mean over folds; SlotHead |
| lexicon audit → LLM label table | 0.899 → 0.906 | **the labels were the bottleneck, not the model** |
| cross-series + RadImageNet arms | 0.910 → 0.920 | multi-arm rank blends over attached checkpoints |
| anatomy-query specialist behind a gate | 0.922 | corpus best |

The last two rows are **not reproducible without their private checkpoint bundles**, and this
notebook does not pretend otherwise: it reproduces everything up to and including the 0.906
label engine plus the 5-fold DINOv2 ensemble, all of it from the mounted competition data.
The blend arms are described in the appendix so a fork with those datasets can add them back.

## The seven decisions that carry the score

1. **Only rank order matters.** AUC reads order alone, so members combine by *rank*-mean, never
   by probability mean, and the submission is written as per-column ranks.
2. **Never flip horizontally.** Four of the twelve targets are `Medial …` or `Lateral …`. A
   mirror swaps them. Left knees are instead *canonicalised* to a right knee once, up front, and
   the augmentation set contains no flip at all.
3. **Laterality comes from the tag; geometry only earns the fallback.** `ImagePositionPatient`
   is a *corner*: walking half the extent along the direction cosines to the image **centre**
   moves tag agreement from 58.8% to 96.9%.
4. **Physical scale, not pixel scale.** A fixed 130 mm crop before the resize makes mm/px
   constant across scanners. 130 mm sits below the field of view of 99.6% of series; 160 mm is
   larger than the image in 60% of them and silently does nothing.
5. **Slices are ordered by geometry.** Projection onto the slice normal. Filename order
   correlates with anatomical order at ρ ≈ 0.01.
6. **Six slots, not one image.** plane × fluid-sensitivity × fat-suppression, with a presence
   mask, so a study missing its coronal T1 is still scored on what it does have.
7. **Two metrics, one selector.** The report-derived holdout picks epochs; the 58 gold labels
   are *reported* and never select. Public LB ≈ gold-58 + 0.044.

## Layout

§1 config · §2–5 the report lexicon · §6 mounts and label sourcing · §7 the pixel contract ·
§8 devices and memory · §9 identifying each series · §10 geometry · §11 the cache · §12 the
model · §13 weights packages and fingerprints · §14 inference and rank-mean · §15 training ·
§16 orchestration · §17 the run guard · appendix.

In [ ]:
# =============================================================================
# 1. Configuration
#
# Every number here was measured somewhere in this repository rather than chosen.
# Where the reason is not obvious it is written down, because a constant with no
# stated reason is indistinguishable from a constant that is wrong.
# =============================================================================
from __future__ import annotations

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import math
import re
import threading
import time
import traceback
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

T0 = time.time()
SEED = 2026


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


# Column order is part of the submission contract, not a convention.
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]
N_TARGET = len(TARGETS)
TARGET_IDX = {t: j for j, t in enumerate(TARGETS)}

# --- what a pixel is ---------------------------------------------------------- #
#
# The centre crop has to be smaller than the smallest field of view in the corpus or it
# silently does nothing. Over every training series the acquired field of view
# (Rows x PixelSpacing) has median 160 mm and runs 70-320 mm: a 160 mm crop is larger
# than the image in 60% of series and is skipped for all of them, leaving their physical
# scale unnormalised. 130 mm is below the field of view of 99.6% of series and still
# contains the joint.
CROP_MM = 130.0

# A feature of width d mm survives resampling only if the pitch is at most d/2. The pitch
# is set by the crop, not by the acquired field of view: CROP_MM / IMG. At 224 px that is
# 0.58 mm, above the 0.5 mm a 1 mm meniscal tear needs; at 336 px it is 0.39 mm and clears
# it. Measured: +0.017 LB for the resize alone.
IMG = 336
IMG_EFFICIENCY = 224       # same weights, cheaper pass; see the appendix

GROUP = 3                  # slices per encoder input, stacked as the three RGB channels
N_GROUP_MAX = 2            # groups per slot the cache will hold if memory allows
SLICE_BAND = (0.20, 0.80)  # fraction of the ordered stack sampled across

# The cache is the one allocation big enough that overshooting it kills the run outright,
# and the encoder, its activations and the pinned batches come out of the same pool. So
# only a fraction of what the machine reports free is taken, and the slice count - the
# cheap, linear axis - gives way before the pixel grid does.
CACHE_FRACTION = 0.42
CACHE_BUDGET_MAX_GB = 22.0
CACHE_BUDGET_GB = 12.0     # only for a machine with no /proc/meminfo
TEST_SHARE = 0.30          # floor on the test corpus relative to train: the visible test
                           # split is a stub and the scored one is not

# --- laterality --------------------------------------------------------------- #
LAT_MIN_OFFSET_MM = 20.0   # inside this the side is not readable from geometry: measured
                           # against the tagged half, the centre-x rule is right 97% of the
                           # time overall and no better than chance within 20 mm
# Gate on the geometry fallback, checked against the studies where both the tag and geometry
# exist. Measured agreement is 96.9%. Using geometry beats leaving the untagged studies
# unnormalised whenever agreement exceeds ~50% - not normalising them mirrors every right
# knee among them - so the gate sits at 0.70: far above chance, far below what is measured,
# and therefore fires only if something structural about the mount has changed.
LAT_MIN_AGREEMENT = 0.70
LAT_WARN_AGREEMENT = 0.90

# --- foreground ---------------------------------------------------------------- #
# The foreground bounding box is used to RECENTRE the fixed physical crop, never to
# resize it. Cropping to the box and resizing would restore per-study scaling and throw
# away the constant mm/px that everything above rests on.
FG_THRESH = 8              # uint8 level above which a voxel is tissue rather than air
FG_MARGIN = 4

# --- I/O ----------------------------------------------------------------------- #
HDR_THREADS = 16
ORDER_THREADS = 32         # slice ordering is latency-bound on the mount, not CPU-bound
PIX_THREADS = 12
# Ceiling, not a target. The ordering pass is ~10^6 small reads over a network mount, so
# its duration is a property of the mount and varies between runs doing identical work.
# Giving up leaves series in file order, which is uncorrelated with anatomy and degrades
# silently - so the ceiling sits well above what the pass ordinarily needs.
ORDER_BUDGET_S = 3600
ORDER_BUDGET_FRAC = 0.25   # ...but never more than this share of the whole run. An absolute
                           # ceiling above the total budget is not a ceiling: it let a 28-min
                           # ordering pass consume a 25-min run and leave decode with nothing.

# --- training ------------------------------------------------------------------ #
N_FOLDS = 5                # rank-mean over folds is worth ~+0.02 and ~1 h: a good trade
                           # against the efficiency track's +0.044 AUC-per-hour break-even
EPOCHS = 10
BATCH_STUDIES = 8          # a study is a bag of up to N_SLOT slot images
EVAL_BATCH = 8
LR_HEAD = 1e-3
LR_BACKBONE = 8e-6         # the encoder is adapted, not retrained
UNFREEZE_LAST = 6          # trainable transformer blocks, from the output end
WEIGHT_DECAY = 0.02
GOLD_WEIGHT = 3.0          # the 58 expert labels, against report-derived ones at <= 1.0
CONF_FLOOR = 0.25          # w = CONF_FLOOR + (1 - CONF_FLOOR) * confidence

# Rigid jitter only. No flip of either axis: the horizontal one swaps medial and lateral,
# which four of the twelve targets name, and the vertical one is anatomically impossible.
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.10

# 9 h is the hard wall. Everything below leaves 40 min of slack for the commit itself.
TIME_BUDGET = 8.3 * 3600
CACHE_BUDGET_FRAC = 0.45   # share of the *remaining* time the two decode passes may spend
CACHE_MIN_S = 20 * 60      # floor under that share. Decode is not the discretionary stage -
                           # folds and epochs are - so if the budget is already gone it still
                           # gets this much and the training loop absorbs the overrun.
CACHE_MIN_FRAC = 0.05      # below this share of studies decoding at least one slot, stop:
                           # the mask is all-absent, every head returns its prior, and the
                           # twelve columns come out constant at exactly the 0.5 benchmark
FINGERPRINT_TOL = 2e-3

# --- slots ---------------------------------------------------------------------- #
# Plane crossed with the acquisition axes. Fat-suppressed fluid-sensitive series exist for
# nearly every study; T1 and non-suppressed fluid-sensitive series are scarcer, which is
# what the presence mask is for.
SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

# Which slots each finding's attention is tilted toward. A fixed table, not a learned
# parameter: it is part of the architecture, so it has to be reproduced exactly for a
# stored state dict to mean anything. Indices are into SLOTS.
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
# Added to the attention logits before the softmax, so it is a tilt and not a mask: at
# 0.55 a favoured slot is e^0.55 = 1.73x as likely to be read. The head can still learn
# past it.
SLOT_PRIOR_STRENGTH = 0.55
POOL = "cls_mean_focal"
POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}

# --- test-time augmentation ----------------------------------------------------- #
# Windows are averaged per target with the pooling its lesion geometry calls for: a
# fracture or a meniscal tear occupies a few slices, so the window that contains it should
# not be diluted by the ones that do not (max); a ligament runs through several (top2);
# a diffuse finding is genuinely everywhere (mean, the default).
TTA_TARGET_POOL = {
    "Fracture": "max", "Contusion": "max",
    "Medial Meniscus": "max", "Lateral Meniscus": "max", "Baker's": "max",
    "ACL": "top2", "MCL": "top2",
}

# --- sequence classification ---------------------------------------------------- #
# GE writes "SAT_GEMS" into ScanOptions, so ScanOptions is matched as exact tokens and
# never as the substring "SAT".
FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")

# --- run mode -------------------------------------------------------------------- #
# "auto": infer from an attached weights package if there is one, otherwise train and then
# infer from what was trained. Overridable for a debug commit.
MODE = os.environ.get("RSNA_MODE", "auto")          # auto | train | infer
SMOKE = os.environ.get("RSNA_SMOKE", "0") == "1"    # tiny end-to-end shakeout
SAVE_WEIGHTS = True        # every trained member is written out with its fingerprint, so
                           # a later inference-only commit can attach this run's output

if SMOKE:
    N_FOLDS, EPOCHS, N_GROUP_MAX = 2, 1, 1
    TIME_BUDGET = 40 * 60

# Folds and epochs are not what makes a run long before the first batch. The header pass, the
# slice-ordering pass and the decode are all proportional to the *corpus*, and on the real
# mount they come to roughly half an hour between them - more than a smoke run's whole budget.
# A shakeout that keeps all 4,407 studies therefore spends everything it has on stage 4 and
# reaches training with an empty cache. So SMOKE caps the corpus too, and keeps every
# gold-labelled study, because the expert-label path is the one worth shaking out.
SMOKE_STUDIES = 128

# An absolute ordering ceiling only means anything relative to the run it sits inside.
ORDER_BUDGET_S = min(ORDER_BUDGET_S, TIME_BUDGET * ORDER_BUDGET_FRAC)

np.random.seed(SEED)
log(f"config: {IMG}px, {CROP_MM:.0f}mm crop ({CROP_MM / IMG:.3f} mm/px, "
    f"{CROP_MM / IMG * 14:.2f} mm per patch token), {N_SLOT} slots, "
    f"{N_FOLDS} folds, mode={MODE}{' SMOKE' if SMOKE else ''}")

In [ ]:
# =============================================================================
# 2. The report lexicon, part 1: text into clauses, and clauses into polarity
#
# 4,349 studies carry a report and no label. This is the machine that turns the first
# into the second. It is the single highest-leverage component in the repository: holding
# the vision model fixed, the move from the naive keyword reading to the audited reading
# below was worth +0.007 LB, and the move from this lexicon to an LLM label table another
# +0.007 on top (see section 5 - if such a table is attached it is preferred, and this
# code becomes the fallback for whatever the table does not cover).
#
# The lexicon's failure mode is silence, not error: on a phrasing it does not carry it
# emits no opinion rather than a wrong one. Silence is measurable without any ground truth
# - for each (report, finding) pair, did anything match? - and that measurement is what
# drove every rule below.
# =============================================================================

# Turkish dotted/dotless i must be folded before casefolding or "IZLENMEZ" and "izlenmez"
# diverge. The German sharp s and the Croatian d-with-stroke likewise.
_PRE = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})


def normalize(text: str) -> str:
    """Fold case, diacritics and separators; keep Greek and Cyrillic letters.

    NFKD strips Latin accents and Greek tonos alike (a-acute -> alpha), which is what we
    want: reports are inconsistent about accents. It also maps MICRO SIGN U+00B5 onto a
    real mu, which matters because most Greek reports here use the wrong codepoint.
    """
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("­", "")                     # soft hyphen
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


def unwrap(text: str) -> str:
    """Rejoin lines a fixed-width layout broke mid-sentence.

    A large share of this corpus arrives hard-wrapped at some column, so a sentence splits
    across two lines with no punctuation at the break. Splitting on newlines then severs
    the finding from its anatomy - "... y parte de la raiz" / "meniscal posterior del
    menisco lateral" puts the tear in one clause and the meniscus in the next, and neither
    says anything alone. A line not ending in sentence punctuation is a continuation.
    """
    if not isinstance(text, str):
        return ""
    out = []
    for line in text.split("\n"):
        s = line.strip()
        if (out and out[-1] and not re.search(r"[.;:!?>*•]$", out[-1])
                and len(out[-1].split()) >= 4 and s and not s[:1].isupper()):
            out[-1] = out[-1] + " " + s
        else:
            out.append(s)
    return "\n".join(out)


_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s+|\n+")


def clauses(text: str):
    """Split into clauses, then attach `header:` lines to the value beneath them.

    A report line reading `Fractures :` followed by `Aucune.` is one statement; splitting
    on punctuation alone separates the anatomy from its negation and flips the label.

    A merged heading must NOT also stand alone. On its own it carries the anatomy word
    with no negation in scope, so `Fractures :` / `Aucune.` asserted a fracture off the
    heading while the joined clause correctly read the denial. The joined clause is a
    superset, so nothing is lost by dropping the bare heading; a heading with no value
    beneath it is not merged and still stands.
    """
    norm = normalize(unwrap(text))
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]
    merged = []
    for i, c in enumerate(raw):
        # Structured English reports write long headings - "lateral compartment (meniscus,
        # collateral ligament complex, cartilage):" is eight words - so the cap is generous.
        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):
            merged.append(c + " " + raw[i + 1])
        else:
            merged.append(c)
    # Comma-separated enumerations inside a long clause hide separate assertions.
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)
    return out


def _rx(*alts: str) -> re.Pattern:
    return re.compile("|".join(alts))


# --------------------------------------------------------------------------- #
# polarity
# --------------------------------------------------------------------------- #
# Negation is scoped by DIRECTION, not by clause. "Subchondral insufficiency fracture at
# the medial tibial plateau without articular surface collapse" contains a negator and
# asserts a fracture: `without` governs what follows it, and the fracture precedes it.
# Reading negation at clause scope turns that sentence into a denial - and it is the house
# style of one of the larger reporting sites here, so the error is systematic.
PRE_NEG = _rx(
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b", r"\babsence\b",
    r"\bno evidence\b", r"\bfree of\b", r"\bnone\b", r"\bneither\b", r"\bnor\b",
    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b", r"\bno se\b",
    r"\bpas de\b", r"\bsans\b", r"\baucune?\b",
    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",
    r"\bkeine?[nmrs]?\b", r"\bohne\b", r"\bnicht\b", r"\bkein\b",
    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",
    r"\bδεν\b", r"\bχωρις\b", r"ουδεν", r"\bουτε\b",
    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",
)

# Turkish and a few Croatian forms put the negator at the end of the sentence, so these
# govern what precedes them instead.
POST_NEG = _rx(
    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",
    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",
    r"izlenmemistir", r"saptanmamistir", r"gorulmemistir", r"\bnema znakova\b",
    r"bez znakova",
)

NEGATION = _rx(PRE_NEG.pattern, POST_NEG.pattern, r"\bunremarkable\b")
NEG_WINDOW = 90


def _negated(clause: str, start: int, end: int) -> bool:
    """True when a negation trigger governs the span [start, end) of this clause."""
    for m in PRE_NEG.finditer(clause):
        if m.end() <= start and start - m.end() <= NEG_WINDOW:
            # "... intact but with a tear" - a contrast conjunction closes the scope.
            if not re.search(r"\b(but|however|ancak|fakat|pero|maar|aber|no i|ali|"
                             r"ωστοσο|αλλα|но)\b", clause[m.end():start]):
                return True
    for m in POST_NEG.finditer(clause):
        if m.start() >= end and m.start() - end <= NEG_WINDOW:
            return True
    # `Heading : negator` - a structured report answers its own heading, so a
    # forward-governing negator standing as the VALUE of a heading still denies the noun in
    # the heading. `Fractures : Aucune.` is the canonical case, and the colon between the
    # two is what distinguishes it from `fracture of the patella, no effusion`, where the
    # negator belongs to a different finding.
    for m in PRE_NEG.finditer(clause):
        if (m.start() >= end and m.start() - end <= NEG_WINDOW
                and ":" in clause[end:m.start()]):
            return True
    return False


NORMALITY = _rx(
    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",
    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",
    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",
    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt", r"\bprimjeren",
    r"\bodrzanog kontinuiteta",
    r"φυσιολογικ", r"ακεραι",
    r"δεν παρατηρουνται", r"δεν σημειωνονται",
    r"unauffallig", r"regelrecht", r"\bo\.?b\.?\b",
    r"нормал", r"запазен", r"съхранен", r"интактн",
    r"\bgaaf\b", r"\bnormaal\b",
)

# Phrases whose whole job is to deny a finding without naming a pathology word. They are
# separated from NORMALITY because they license a negative even when the clause contains
# no pathology term at all, which is how a report clears a structure in one line.
NORMAL_PHRASE = _rx(
    r"\bsin alteracion", r"\bsin cambios\b", r"\bsin particularidad",
    r"\bsin hallazgos\b", r"\bsin lesion", r"\bsin signos de (rotura|lesion)",
    r"\bcontinu[oa]s?\b", r"\bcontinuidad conservada\b",
    r"\bno abnormalit", r"\bno significant abnormalit", r"\bunremarkable\b",
    r"\bno evidence of (tear|injury|abnormalit)",
    r"\bohne auffalligkeit", r"\bkein nachweis\b", r"\bohne befund\b",
    r"\bgeen afwijking", r"\bzonder afwijking",
    r"\bsans anomalie", r"\bpas d[e']anomalie",
    r"\bbez osobitosti\b", r"\bbez znakova (rupture|lezije)\b", r"\bbez patoloskih\b",
    r"χωρις αλλοιωσ", r"χωρις παθολογ",
    r"δεν παρατηρουνται (αξιολογα|παθολογ)",
    r"\bбез особености\b", r"\bбез патологич", r"\bбез данни за\b",
    r"\bozel bir ozellik yok", r"\bpatolojik bulgu (yok|izlenmemis)",
)

UNCERTAIN = _rx(
    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",
    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",
    r"\br/o\b", r"\blikely\b", r"\bsuggest", r"\bcompatible with\b",
    r"\bposible\b", r"sin criterios categoricos", r"\bdudos", r"\bsugier",
    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim", r"\bdusundur",
    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b", r"\bmoze odgovarati\b",
    r"πιθαν", r"υποπτ",
    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bv\.?a\.?\b", r"\bwohl\b",
    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",
    r"\bmogelijk\b", r"\bverdacht\b",
)

In [ ]:
# =============================================================================
# 3. The report lexicon, part 2: pathology, grades, anatomy
# =============================================================================

TEAR = _rx(
    r"\btear", r"\btorn\b", r"\brupture", r"\bdisruption\b", r"discontinuit",
    r"\bavuls", r"\bmacerat", r"\bbuckethandle\b", r"bucket handle",
    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",
    r"\bdechirure", r"\bdechire",
    r"\bscheur", r"\bruptuur", r"gescheurd",
    r"\briss\b", r"einriss", r"\bruptur", r"zerreiss", r"\blasion", r"\bausriss",
    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi", r"\brupturu\b",
    r"devamsizlik", r"\brupture\b", r"\bdevamliligi secilememis",
    r"\bpuknuce", r"\bprekid\b", r"\bpukotin", r"\bruptur",
    r"ρηξη", r"ρηξις", r"ρηγμα", r"ασυνεχεια",
    r"руптура", r"разкъсв", r"разрив", r"скъсв", r"\bлезия\b",
)

DEGEN = _rx(
    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",
    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλ", r"дегенерат",
    r"\bμυξοειδ", r"\bμυξωδ", r"\bmeniskopat", r"\bmeniscopath",
    r"\bmuco ?ide\b", r"aufgefasert", r"\bdejenerasyon\b",
)

INJURY = _rx(
    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",
    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",
    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",
    r"aumento de senal", r"alteracion de senal", r"cambio de senal",
    r"\bsignalanhebung", r"\bsignalalteration", r"verhoogd signaal", r"sinyal artis",
    r"αυξημενο σημα", r"повишен сигнал", r"\besguince\b",
    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",
    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",
    r"\bpartiel", r"\bpartiell",
)

# A numeric grade is the most precise thing a knee report says, and it means different
# things in different places: grade 3 of a meniscus is a tear by definition, grade 1 or 2
# is intrasubstance signal that never reaches the surface and is not one. A ligament runs
# the other way round - grade 1 is a stretch, grade 2 a partial tear. So the grade is read
# as a number and interpreted per structure rather than folded into one pathology
# vocabulary. This distinction alone moved several hundred meniscus labels.
_GRADE_RX = re.compile(
    r"(?:grade|grad|grado|grau|derece|stupnja|stupanj|βαθμ|степен|icrs|outerbridge)"
    r"[\s:]*(?:grade\s*)?([1-4]|iv|iii|ii|i)\b"
)
_ROMAN = {"i": 1, "ii": 2, "iii": 3, "iv": 4}


def _grade_of(clause: str):
    """Highest numeric grade stated in a clause, or None."""
    best = None
    for m in _GRADE_RX.finditer(clause):
        v = m.group(1)
        n = _ROMAN.get(v, None) if not v.isdigit() else int(v)
        if n is not None and (best is None or n > best):
            best = n
    return best


GRADE_HIGH = re.compile(r"grade?[ao]?\s*(3|4|iii|iv)\b|icrs grade (iii|iv|3|4)|"
                        r"stupnja iv|stupnja iii|\bgrado (3|4)\b|\bgrad (3|4)\b|"
                        r"\bgrade (3|4)\b")

# --------------------------------------------------------------------------- #
# anatomy: the four side-specific structures
# --------------------------------------------------------------------------- #
# Nine languages. The plural forms matter as much as the singular ones: a report that
# clears both cruciates in one phrase ("kreuzbander regelrecht") says nothing about either
# one under a singular-only lexicon, and those blanket clearances are the single largest
# source of true negatives in the corpus.
ANAT = {
    "ACL": _rx(
        r"anterior cruciate", r"\bacl\b",
        r"cruzado anterior", r"\blca\b",
        r"croise anterieur",
        r"voorste kruisband", r"\bvkb\b",
        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",
        r"on capraz", r"\bocb\b", r"anterior capraz",
        r"prednji krizni", r"prednjeg krizn",
        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",
        r"\bχιαστ\w*",
        r"предна кръстна", r"предната кръстна", r"предна кръста",
        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",
        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",
        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",
    ),
    "MCL": _rx(
        r"medial collateral", r"\bmcl\b", r"tibial collateral",
        r"colateral medial", r"colateral interno", r"\blcm\b",
        r"collateral medial", r"collateral interne",
        r"mediale collaterale", r"binnenband", r"\b(mediale|laterale) banden\b",
        r"\bcollaterale banden\b",
        r"innenband", r"mediales? kollateral",
        r"\bic yan bag", r"medial kollateral", r"\biyb\b", r"medyal kollateral",
        r"medijalni kolateraln", r"medijalnog kolateraln",
        r"εσω πλαγι", r"εσωτερικο πλαγι", r"\bπλαγι\w* συνδεσμ", r"\bπλαγιοι\b",
        r"медиален колатерал", r"вътрешна странична", r"\bколатерал\w*",
        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",
        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",
        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",
        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",
        r"колатерални връзки", r"страничните връзки",
    ),
    "Medial Meniscus": _rx(
        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",
        r"menisco medial", r"menisco interno",
        r"menisque medial", r"menisque interne",
        r"mediale meniscus", r"binnenmeniscus",
        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",
        r"medyal menisk", r"\bic menisk",
        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",
        r"medijaln\w* menisk\w*", r"\bmedijalnog meniska\b", r"medijalni menisk",
        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",
        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",
        r"oba meniska", r"both menisci", r"ambos meniscos", r"beide menisci",
        r"her iki menisku", r"αμφοτερ\w* μηνισκ",
        r"двата менискуса", r"medial (and|&) lateral menisc",
    ),
    "Lateral Meniscus": _rx(
        r"lateral meniscus", r"lateral menisc",
        r"menisco lateral", r"menisco externo",
        r"menisque lateral", r"menisque externe",
        r"laterale meniscus", r"buitenmeniscus",
        r"aussenmeniskus", r"lateralen? meniskus", r"aussenmeniskushinterhorn",
        r"lateral menisk", r"\bdis menisk",
        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",
        r"lateraln\w* menisk\w*", r"\blateralnog meniska\b",
        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",
        r"латералния менискус", r"латерален менискус", r"външния менискус",
        r"oba meniska", r"both menisci", r"ambos meniscos", r"beide menisci",
        r"her iki menisku", r"αμφοτερ\w* μηνισκ",
        r"двата менискуса", r"medial (and|&) lateral menisc",
    ),
}

PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_TARGETS = ["Medial OA", "Lateral OA", "PF OA"]

# Osteoarthritis is written as cartilage damage far more often than as a diagnosis, so the
# evidence lexicon is deliberately much wider than the word.
OA_EVIDENCE = _rx(
    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",
    r"chondropath", r"chondromalac", r"condropat", r"condromalac", r"\bchondros",
    r"\bchondrosis\b", r"chondral (loss|defect|ulcer|thinning|injury|fissur|wear)",
    r"cartilage (loss|thinning|defect|fissur|wear|damage|heterogeneity|irregularit)",
    r"(loss|thinning|fissur|defect|ulcer|erosion|denudation) of[^.]{0,20}cartilage",
    r"articular cartilage[^.]{0,30}(loss|thin|fissur|defect|erosion|wear|irregular)",
    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten", r"spurring",
    r"joint space narrowing", r"pinzamiento articular", r"reduced joint space",
    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral", r"kikirdak dejener",
    r"eklem aralig\w* daral", r"eklem mesafesi daral", r"kikirdak kalinlig\w* azal",
    r"kraakbeen", r"gonartrose", r"artrose", r"\bknorpel", r"arthrose", r"gonarthrose",
    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit", r"artrotsk", r"artrotick",
    r"\boa promjen", r"\boa\b", r"degenerativne promjene hrskav",
    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ", r"χονδρομαλακ",
    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου", r"διαβρωση του αρθρικου χονδρ",
    r"λεπτυνση[^.]{0,30}χονδρ", r"φθορα[^.]{0,20}χονδρ",
    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,40}(изтън|увред|дефект|липс)",
    r"изтъняване[^.]{0,30}хрущял", r"хондромалац",
    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",
    r"icrs grade", r"icrs\b", r"outerbridge", r"\bdenudation\b", r"denudacij",
    r"erozivne promjene", r"\berosion of[^.]{0,20}cartilage",
    r"kraakbeenlijden", r"kraakbeenverlies",
)

# --------------------------------------------------------------------------- #
# where in the joint a cartilage statement sits
# --------------------------------------------------------------------------- #
# Three of the twelve targets are compartments, and a report almost never writes "medial
# OA". It writes "chondral thinning of the medial femoral condyle". Attributing cartilage
# statements to a compartment is therefore a localisation problem, not a keyword problem.
#
# TF_SITE is consulted only inside a clause that already carries OA evidence, so a bare
# "condyle" or "femur" is safe here and would not be safe anywhere else.
TF_SITE = _rx(
    r"compartment", r"compartimento", r"compartiment", r"kompartman", r"kompartiment",
    r"kompartment", r"odjelj", r"διαμερισμα", r"компартм", r"\bотдел",
    r"femorotibial", r"tibiofemoral", r"femoro tibial", r"femorotibiaal",
    r"femorotibijaln", r"феморотибиал", r"\bft zglob", r"tibiofemoraln",
    r"condyle", r"condilo", r"kondyl", r"kondil", r"condyl", r"κονδυλ",
    r"кондил", r"\bplateau", r"\bplato\b", r"platillo", r"meseta", r"плато",
    r"tibiaplateau", r"tibijaln\w* plato", r"tibyal plato", r"tibia plato",
    r"κνημιαι", r"μηριαι", r"weightbearing", r"weightbaring", r"zona de carga",
    r"dragende deel", r"agirlik tasiyan", r"\bfemur\b", r"\btibia\b", r"\bfemoral\b",
    r"\btibial\b", r"\bfemura\b", r"\btibije\b", r"\bmesarthrio\b", r"μεσαρθριο",
)

PF_SITE = _rx(
    r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",
    r"retropatellar", r"retrorotulian", r"trochlea", r"troclea", r"troklea",
    r"trochlear", r"trohlej", r"τροχιλ", r"\bpatella", r"\bpatellar", r"rotulian",
    r"\brotula\b", r"\bpatele\b", r"patellofemoraal", r"femoropatellair",
    r"επιγονατιδ", r"μηροεπιγονατιδ", r"пател", r"феморопател",
    r"anterior compartment", r"compartimento anterior", r"prednj\w* odjeljk",
    r"\bfp zglob", r"\bpf zglob", r"\bfaset", r"\bfacet", r"patellofemoraln",
)

SIDE_MEDIAL = _rx(
    r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",
    r"\bmediale\w*", r"\binterno\b", r"\binterna\b", r"\binternos\b", r"\binterne\b",
    r"\binnen\w*", r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",
    r"\bмедиал\w*", r"\bвътреш\w*", r"\bbinnen\w*", r"\bmediaal\b", r"\bmediales?\b",
)
SIDE_LATERAL = _rx(
    r"\blateral\w*", r"\bexterno\b", r"\bexterna\b", r"\bexternos\b", r"\bexterne\b",
    r"\bdis\b", r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",
    r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*", r"\bvanjsk\w*",
)
SIDE_ANTERIOR = _rx(
    r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*", r"\bvorder\w*",
    r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*", r"\banteriyor\w*", r"\bavant\b",
    r"\banterieur\w*",
)
ANY_SIDE = _rx(SIDE_MEDIAL.pattern, SIDE_LATERAL.pattern)

# A whole-joint diagnosis. Nothing localises it, but it implicates every compartment -
# "gonarthrosis" is by definition not confined to one. These statements are what the
# inheritance rule in _score_oa exists to spend.
GLOBAL_OA = _rx(
    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",
    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose", r"gonartro",
    r"goanrtrot", r"gonartrot",
    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",
    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",
    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος", r"εκφυλιστικη οστεοαρθριτ",
    r"артроза на колянната", r"гонартроз",
    r"degenerative joint disease", r"\bdjd\b", r"three compartments",
    r"compartmens", r"compartments",
)

In [ ]:
# =============================================================================
# 4. The report lexicon, part 3: self-declaring findings, stems, severity
# =============================================================================

# These five findings name themselves - there is no anatomy word to pair with a pathology
# word, the term IS the finding. So they are matched directly and only scoped for negation.
DIRECT = {
    "Effusion": _rx(
        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",
        r"\bhemarthros", r"\bhaemarthros",
        r"derrame articular", r"\bderrame\b", r"liquido articular", r"hemartrosis",
        r"epanchement",
        r"gewrichtsvocht", r"\bvocht\b", r"gewrichtseffusie", r"opzetting van suprapatell",
        r"gelenkerguss", r"\berguss\b", r"gelenksergu", r"gelenksflussigkeit",
        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi", r"eklem mesafesinde sivi",
        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",
        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",
        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",
        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",
        r"συλλογη υγρου", r"ενθαρθρικ",
        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",
    ),
    "Synovitis": _rx(
        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",
        r"thicken\w* synovial", r"hypertroph\w* of the synovium",
        r"synoviale? (verdikking|proliferatie)", r"verdikkingen van (het )?synovium",
        r"synovialitis", r"synovialis(verdickung|proliferation)", r"reizsynovial",
        r"sinovijalitis", r"sinovitis", r"zadebljanje sinovij", r"proliferacij\w* sinovij",
        r"sinovijaln\w* proliferacij",
        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",
        r"παχυνση[^.]{0,20}υμεν", r"υμενα",
        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",
        r"\bpannus\b", r"\bhoffit", r"sinovyal\w* (kalinlas|proliferas)",
        r"sinovyal hipertrof", r"\bartrit\b", r"\barthritis\b",
    ),
    "Baker's": _rx(
        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",
        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",
        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist", r"popliteal\w* cist",
        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",
        r"συνοβιακη κυστη", r"κυστη τυπου baker",
        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста", r"бекеров",
        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",
    ),
    "Contusion": _rx(
        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",
        r"marrow o?edema", r"\bkontuz", r"medular bone o?edema", r"osseous contusion",
        r"contusion osea", r"edema oseo", r"edema de medula osea", r"contusiones oseas",
        r"oedeme osseux", r"contusion osseuse",
        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",
        r"knochenmarkodem", r"knochenodem", r"knochenmarksodem", r"kontusion",
        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi", r"kemik iliginde odem",
        r"kontuzyonel kemik", r"kemik iligi odemleri",
        r"kostani edem", r"edem kosti", r"kontuzij", r"kostane srzi[^.]{0,20}edem",
        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα", r"οστικο μωλωπ",
        r"костномозъчен едем", r"костен едем", r"контузионен", r"костно мозъчен едем",
    ),
    "Fracture": _rx(
        r"\bfractur", r"\bfract\b",
        r"\bfractura", r"\bfracturas\b",
        r"\bfractuur", r"\bbreuk\b",
        r"\bfraktur", r"\bbruch\b",
        r"\bkirik\b", r"\bkirigi\b", r"\bkiri[kg]\w*",
        r"\bprijelom", r"impresijsk[^ ]* fraktur", r"impaktcij",
        r"καταγμα", r"καταγματ",
        r"фрактур", r"счупван", r"фисур",
        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",
        r"subchondral fracture", r"subkondral kiri", r"impaction (fracture|injury)",
        r"osteochondral (fracture|impaction)", r"\bsegond\b", r"impactiefractuur",
        r"subchondrale impression", r"subchondraler? impress",
    ),
}

# Phrases that contain a finding's word while denying that finding. "Microfracture" is a
# surgical technique, and "fracture risk" is a prediction, not a fracture; "meniscal cyst"
# is not a Baker's cyst even though both are popliteal fluid collections.
DECOY = {
    "Fracture": _rx(r"microfractur", r"\bfracture (risk|prophyla)"),
    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"parameniscal"),
}

# --------------------------------------------------------------------------- #
# stems, for morphology the phrase lexicons cannot reach
# --------------------------------------------------------------------------- #
# Nine languages of inflection cannot be enumerated as phrases. A stem plus a
# side-qualifier within a short window generalises over case endings, compounding and
# word order without the false positives a bare stem would bring.
STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")
STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",
                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",
                    r"\bacl\b", r"\blca\b", r"\bvkb\b", r"\bocb\b", r"\bacb\b")
STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",
                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",
                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",
                      r"innenband\w*", r"binnenband\w*", r"\bmcl\b", r"\blcm\b",
                      r"\biyb\b")
STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",
                    r"kiri[kg]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",
                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",
                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")

# A report that clears or tears both menisci in one breath - "normaal voorkomen menisci",
# "normal medial and lateral menisci" - names neither side, so a side-qualified lexicon
# leaves both targets silent. Consulted ONLY when the clause names no side at all;
# otherwise "tear of the medial meniscus, menisci otherwise intact" would fire the plural
# for the lateral side off a clause that is about the medial one.
PLURAL_MENISCI = _rx(
    r"\bmenisci\b", r"\bmeniscos\b", r"\bmenisques\b", r"\bmenisken\b",
    r"\bmeniskusi\b", r"\bmenisk\w*ler\b", r"\bμηνισκοι\b", r"\bμηνισκων\b",
    r"\bменискуси\b", r"\bменискусите\b", r"\bmenisci\w*\b",
)

# Structures that steal a cruciate / collateral stem match for the wrong ligament. The
# competition scores the ACL and the MCL; the PCL and the LCL are in every report and in
# neither label.
POSTERIOR_ONLY = _rx(r"\bpcl\b", r"\blcp\b", r"\bhkb\b", r"\bacb\b",
                     r"posterior cruciate", r"cruzado posterior", r"croise posterieur",
                     r"achterste kruisband", r"hinteres kreuzband", r"arka capraz",
                     r"straznji krizn", r"οπισθι[οα]\w* χιαστ", r"задна кръстн",
                     r"задната кръстн")
LATERAL_COLL_ONLY = _rx(r"\blcl\b", r"\bfcl\b", r"lateral collateral",
                        r"fibular collateral", r"colateral lateral", r"colateral externo",
                        r"buitenband", r"aussenband", r"dis yan bag",
                        r"lateralni kolateraln", r"εξω πλαγι", r"латерален колатерал")


def _distance(clause: str, a: re.Pattern, b: re.Pattern):
    """Shortest gap in characters between any match of `a` and any match of `b`."""
    best = None
    for ma in a.finditer(clause):
        for mb in b.finditer(clause):
            d = max(mb.start() - ma.end(), ma.start() - mb.end(), 0)
            if best is None or d < best:
                best = d
    return best


def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    """True when a qualifier sits within `window` characters of a stem match.

    55 characters is roughly one clinical phrase in any of these languages. Wider and
    "medial meniscus ... lateral collateral ligament" cross-links; narrower and normal
    German and Croatian compounding falls outside it.
    """
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        if qual_rx.search(clause[lo:hi]):
            return True
    return False


STEM_RULES = {
    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),
    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),
    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),
    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),
}


class _Matcher:
    """Phrase lexicon first; stem-plus-qualifier as the fallback, never the other way.

    The phrase form is precise and its match position is meaningful; the stem form is a
    recall net. Trying the stem first would let a loose match shadow an exact one and
    would move the span the negation window is measured from.
    """

    def __init__(self, phrase_rx, stem=None, side=None, window=55, contrary=None):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window
        self.contrary = contrary

    def _wrong_side(self, clause, m):
        """A stem hit inside a phrase naming the OTHER structure is not our structure."""
        if self.contrary is None or m is None:
            return False
        return _near(clause, self.stem, self.contrary, self.window) and \
            self.phrase_rx.search(clause) is None

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None:
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            m = self.stem.search(clause)
            if not self._wrong_side(clause, m):
                return m
        return None


# "posterior cruciate" contains the cruciate stem and is not the ACL; "lateral collateral"
# contains the collateral stem and is not the MCL. Both are named in nearly every complete
# report, so without this the two ligament targets inherit the other ligament's findings.
CONTRARY = {"ACL": POSTERIOR_ONLY, "MCL": LATERAL_COLL_ONLY}

ANAT_MATCH = {
    t: _Matcher(ANAT[t], *STEM_RULES[t], contrary=CONTRARY.get(t))
    for t in PAIRED
}
DIRECT_MATCH = {
    t: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if t == "Fracture" else rx)
    for t, rx in DIRECT.items()
}

# --------------------------------------------------------------------------- #
# severity
# --------------------------------------------------------------------------- #
# Severity does not change the sign of a label, only its magnitude, which is exactly what
# a soft target wants: "trace effusion" and "large tense effusion" are both positive and
# the annotator did not score them alike.
SEV_LOW = _rx(
    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",
    r"\btiny\b", r"\bscant\b", r"\bdiscrete\b", r"\blow ?grade\b", r"\bincipient\b",
    r"\bleve\b", r"\bminim", r"\bpeque", r"\bfina\b", r"\bfino\b", r"\bligero\b",
    r"\bescaso\b", r"\bdiscreto\b",
    r"\bhafif\b", r"\baz miktarda\b", r"\bsilik\b",
    r"\bmanj\w*", r"\bblago\b", r"\bdiskretn", r"\bmalo\b", r"\bpocetn",
    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",
    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b", r"\blicht\b",
    r"\bηπι", r"\bμικρ", r"\bελαχιστ", r"\bαρχομεν",
    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",
)

SEV_HIGH = _rx(
    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",
    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",
    r"\bcomplete\b", r"\bfull ?thickness\b", r"\bhigh ?grade\b", r"\badvanced\b",
    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",
    r"\bespesor total\b", r"\bcompleta?\b",
    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b", r"\bkomplet",
    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",
    r"\buznapredoval", r"\bpotpun", r"\bkompleksn",
    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",
    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b", r"\bvolledig",
    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ", r"\bπληρη",
    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен", r"\bпълн",
)

# Marrow signal that is degenerative rather than traumatic. "Subchondral cystic change"
# next to an osteophyte is chronic and is not the Contusion label; the same signal
# described with a mechanism is.
DEGENERATIVE_MARROW = _rx(
    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",
    r"υποχονδρι", r"υπαρθρικ", r"субхондрал", r"subchondrale?", r"subartikuler",
    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",
    r"degenerative", r"degenerativ", r"reaktiv", r"\bcisti\b",
)

TRAUMA = _rx(
    r"\bbruise\b", r"\bcontusion", r"\bkontuz", r"\btrauma", r"\bimpaction\b",
    r"\bpivot shift\b", r"\bkissing\b", r"\bacute\b", r"\bagudo\b", r"\bakut",
    r"\bpivot kaymasi\b", r"\bcontusion osseuse\b", r"\bbone bruise\b",
    r"\bbotcontusie\b", r"\bконтузион", r"\bμωλωπ", r"\bkontuzij", r"\bimpaktcij",
    r"\bimpakcij", r"\bfall\b", r"\binjury\b", r"\bimpression\b",
)

# Effusion-adjacent inflammatory signs, used only when a report never names synovitis.
# 88% of reports never write the word; the annotator marked it on roughly half the
# studies. Silence there is a reporting convention, not a negative finding.
SYNOVIAL_PROXY = _rx(
    r"bursit", r"burzit", r"\bbursa\b[^.]{0,30}(fluid|distend|sivi|tekucin|opzetting)",
    r"suprapatellar (bursitis|effusion|recess)", r"suprapatellar bursa",
    r"suprapatellar bursada", r"suprapatelarno", r"suprapatellaire recessus",
    r"hoffa", r"hoffit", r"plica", r"plika", r"πλικα", r"fat pad[^.]{0,20}(edema|oedema)",
    r"kapsul", r"capsul", r"καψ", r"капсул", r"\bpannus\b", r"\bsinov", r"\bsynov",
)

In [ ]:
# =============================================================================
# 5. The report lexicon, part 4: counting evidence into a soft label
#
# Every target ends as a (score, confidence) pair. The score is the training target; the
# confidence becomes the sample weight, so a finding the report states plainly pulls
# harder than one inferred from context, and a finding the report never mentions barely
# pulls at all. That weighting is what lets 4,349 noisy labels train twelve heads.
# =============================================================================

# The five rules that separate the audited reading (LB 0.906) from the naive one (0.891).
# Each is a switch so a fork can measure it: flip one to False, rerun the coverage gauge,
# and see what it was worth.
FEATURES = {
    "unwrap": True,                 # rejoin hard-wrapped sentences before splitting
    "directional_negation": True,   # negation scoped by direction, not by clause
    "oa_inherit": True,             # whole-joint OA implicates uncleaned compartments
    "graded_pathology": True,       # grade read on the right scale per structure
    "synovitis_backoff": True,      # inflammatory proxies when the word never appears
}


def _polarity(clause: str, span=None) -> str:
    """positive / negative / uncertain for a term matched at `span` in this clause."""
    if UNCERTAIN.search(clause):
        return "uncertain"
    if span is None or not FEATURES["directional_negation"]:
        if NEGATION.search(clause):
            return "negative"
    elif _negated(clause, span[0], span[1]):
        return "negative"
    if NORMALITY.search(clause):
        # `meniscus normal` denies; `normal alignment ... full thickness tear` does not.
        if TEAR.search(clause) or GRADE_HIGH.search(clause):
            return "positive"
        return "negative"
    return "positive"


def _severity(clause: str) -> float:
    """How emphatic a clause is about the finding it asserts.

    A numeric grade is deliberately not read here. It belongs to whichever structure the
    grade was written for, and a clause that grades the cartilage while mentioning the
    effusion in passing - `PF arthrotic change with reduced joint space, a smaller effusion
    and grade IV chondromalacia` - would otherwise report a severe effusion.
    """
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and not low:
        return 1.0
    if low and not high:
        return 0.45
    if high and low:
        return 0.8
    return 0.75


def _grade(n_pos, n_neg, n_unc, best):
    """Map counted evidence onto a score in (0, 1) and a confidence.

    Positives never reach 1.0 and negatives never reach 0.0: these are soft targets under
    a BCE loss, and a saturated target contributes an unbounded gradient off a label that
    came out of a regex. Silence lands at 0.28 with confidence 0.05 - close to the corpus
    prevalence, weighted almost out of the loss.
    """
    if n_pos or n_unc:
        score = min(0.97, 0.50 + 0.45 * best + 0.015 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.20 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = 0.28, 0.05
    return score, conf


def _paired_weight(clause: str, meniscus: bool) -> float:
    """How strongly one clause asserts damage to a meniscus or a cruciate/collateral.

    Ordered, not calibrated. What has to hold is that a tear outranks a graded lesion, that
    the grade is read on the right scale for the structure, and that intrasubstance
    degeneration lands below both - the annotator marks a torn meniscus and leaves a
    degenerate one, and a lexicon that scores the two alike throws that ordering away.
    """
    g = _grade_of(clause) if FEATURES["graded_pathology"] else None
    tear = TEAR.search(clause) is not None
    if meniscus:
        if tear:
            base = 1.0
        elif g is not None:
            base = 0.95 if g >= 3 else 0.30    # grade 3 reaches the surface; 1-2 does not
        elif DEGEN.search(clause):
            base = 0.35
        else:
            base = 0.45
    else:
        if tear:
            base = 1.0
        elif g is not None:
            base = 0.85 if g >= 2 else 0.30    # ligament grade 2 is already a partial tear
        elif DEGEN.search(clause):
            base = 0.40
        else:
            base = 0.55
    if SEV_HIGH.search(clause) and not SEV_LOW.search(clause):
        base = min(1.0, base * 1.2)
    elif SEV_LOW.search(clause) and not SEV_HIGH.search(clause):
        base *= 0.7
    return base


def _score_paired(cls, tgt):
    """Evidence for one of the four side-specific ligament / meniscus targets."""
    anat_rx = ANAT_MATCH[tgt]
    path_rx = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)
    meniscus = "Meniscus" in tgt
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        hit = anat_rx.search(c)
        if hit is None and meniscus and PLURAL_MENISCI.search(c) \
                and not ANY_SIDE.search(c):
            hit = PLURAL_MENISCI.search(c)
        if hit is None:
            continue
        # Anchor the negation test on the pathology word, not on the anatomy word.
        # `Medial meniscus is not torn` negates the tear, and the negator stands after the
        # noun: scoping from the noun would read the sentence as an assertion.
        pm = path_rx.search(c)
        if pm is None and _grade_of(c) is None:
            # The structure is named with no pathology word at all. That is how a report
            # clears it - "ACL intact", "menisci unremarkable" - and it is a negative.
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c)
                                           and not NEGATION.search(c)):
                n_neg += 1
            continue
        span = (pm.start(), pm.end()) if pm is not None else None
        pol = _polarity(c, span)
        if pol == "positive":
            n_pos += 1
            best = max(best, _paired_weight(c, meniscus))
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.45 * _paired_weight(c, meniscus))
    s, cf = _grade(n_pos, n_neg, n_unc, best)
    return s, cf, n_pos, n_neg


def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None,
                   context_bonus=None):
    """Evidence for a self-declaring finding, where the term is the diagnosis."""
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and not path_rx.search(c):
            if NORMAL_PHRASE.search(c) or (NORMALITY.search(c)
                                           and not NEGATION.search(c)):
                n_neg += 1
            continue
        pol = _polarity(c, (m.start(), m.end()))
        if pol == "positive":
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.30)
    s, c = _grade(n_pos, n_neg, n_unc, best)
    return s, c, n_pos, n_neg


def _score_oa(cls):
    """Osteoarthritis, scoped to the three compartments.

    A cartilage statement is attributed by what else the clause names rather than by a
    compartment phrase: `medial` next to a tibiofemoral structure sends it to the medial
    compartment, `patella` or `trochlea` to the patellofemoral one, and a statement that
    names neither is a whole-joint assertion that counts for all three at a discount.
    That last case is the majority of the corpus - reports say `tricompartmental
    chondrosis`, not `chondrosis of the medial femorotibial compartment` - and leaving it
    silent is what capped the three OA targets in earlier revisions.
    """
    acc = {t: {"pos": 0, "neg": 0, "unc": 0, "best": 0.0} for t in OA_TARGETS}
    g_pos, g_neg, g_best = 0, 0, 0.0

    for c in cls:
        m = OA_EVIDENCE.search(c)
        if not m:
            continue
        pol = _polarity(c, (m.start(), m.end()))
        sev = _severity(c)
        tf_med = _near(c, TF_SITE, SIDE_MEDIAL, 45)
        tf_lat = _near(c, TF_SITE, SIDE_LATERAL, 45)
        pf = PF_SITE.search(c) is not None
        hits = []
        if tf_med:
            hits.append("Medial OA")
        if tf_lat:
            hits.append("Lateral OA")
        if pf:
            hits.append("PF OA")

        if not hits:
            # No compartment named. Whole-joint statements ("tricompartmental chondrosis",
            # "gonarthrose") speak for all three; an unlocalised cartilage remark is weaker
            # evidence but still evidence, so it is carried at a discount, not dropped.
            if pol == "positive":
                g_pos += 1
                g_best = max(g_best, sev if GLOBAL_OA.search(c) else sev * 0.7)
            elif pol == "negative":
                g_neg += 1
            continue

        for t in hits:
            if pol == "positive":
                acc[t]["pos"] += 1
                acc[t]["best"] = max(acc[t]["best"], sev)
            elif pol == "negative":
                acc[t]["neg"] += 1
            else:
                acc[t]["unc"] += 1
                acc[t]["best"] = max(acc[t]["best"], 0.30)

    out = {}
    for t in OA_TARGETS:
        a = acc[t]
        pos, neg, unc, best = a["pos"], a["neg"], a["unc"], a["best"]
        if not (pos or unc) and g_pos and FEATURES["oa_inherit"]:
            # Inherit the whole-joint statement, unless this compartment was separately
            # and explicitly cleared - in which case the report contradicts itself, and
            # the local statement wins but not by the full margin.
            if neg:
                score, conf = _grade(0, neg, 0, 0.0)
                score = max(score, 0.35)
                conf *= 0.7
            else:
                score, conf = _grade(g_pos, 0, 0, g_best * 0.92)
                conf *= 0.75
        else:
            score, conf = _grade(pos, neg + g_neg, unc, best)
        out[t] = (score, conf, pos, neg)
    return out


def extract(report: str) -> dict:
    """Twelve (score, confidence) pairs, plus the counts the coverage gauge reads."""
    cls = clauses(report)
    out = {}

    for tgt in PAIRED:
        s, c, npos, nneg = _score_paired(cls, tgt)
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    for tgt, (s, c, npos, nneg) in _score_oa(cls).items():
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    for tgt in ("Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"):
        if tgt == "Contusion":
            # Marrow oedema is the same words whether it is a bruise or a chronic
            # subchondral change; the surrounding clause is the only way to tell.
            s, c, npos, nneg = _score_clauses(
                cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt),
                context_penalty=DEGENERATIVE_MARROW, context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None,
                                              DECOY.get(tgt))
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    # --- synovitis backoff --------------------------------------------------- #
    # Eighty-eight per cent of reports never write the word, and the annotator marks it on
    # nearly half the studies: the label cannot be read off the term alone. What the report
    # does say is whether the joint is wet and irritated - an effusion, a distended bursa,
    # an inflamed fat pad - and that ordering is the only signal available on the silent
    # majority. It enters at confidence 0.18, so it shapes the ranking without asserting a
    # finding, and it never overrides a report that does discuss synovium.
    if (FEATURES["synovitis_backoff"] and out["Synovitis__npos"] == 0
            and out["Synovitis__nneg"] == 0):
        proxy = sum(1 for c in cls if SYNOVIAL_PROXY.search(c)
                    and _polarity(c) == "positive")
        eff = out["Effusion"]
        prior = 0.30 + 0.30 * max(0.0, (eff - 0.5) / 0.45) + 0.06 * min(proxy, 3)
        out["Synovitis"] = min(0.72, prior)
        out["Synovitis__conf"] = 0.18

    return out


# --- a self-test, because a silent regex failure is invisible ----------------- #
# These are the cases that were actually wrong at some point in this repository. Each pins
# one rule, and each was a real label flip on real studies before it was fixed. A lexicon
# this size cannot be reviewed by reading it, so it is pinned instead.
_LEXICON_CASES = [
    # directional negation: `without` governs forward, so the fracture before it stands
    ("Subchondral insufficiency fracture of the medial tibial plateau without "
     "articular surface collapse.", "Fracture", "pos"),
    # heading merge plus the colon rule: the denial is the value of the heading
    ("Fractures :\nAucune.", "Fracture", "neg"),
    # Turkish sentence-final negation governs backwards
    ("Medial menisküste yırtık izlenmemektedir.", "Medial Meniscus", "neg"),
    # the PCL must not answer for the ACL
    ("Posterior cruciate ligament is completely torn.", "ACL", "neg"),
    # whole-joint OA is inherited by a compartment the report never localises
    ("Tricompartmental gonarthrosis with marginal osteophytes.", "Lateral OA", "pos"),
    # a blanket clearance names no side and must still clear both menisci
    ("Kreuzbänder regelrecht. Menisken unauffällig.", "Lateral Meniscus", "neg"),
    # unwrap: the tear and the meniscus it belongs to are on different lines
    ("Se observa una rotura horizontal en el cuerpo\ny parte de la raíz posterior del "
     "menisco medial.", "Medial Meniscus", "pos"),
]

# Ordering claims, which matter more than any absolute value under an AUC metric.
_LEXICON_ORDER = [
    # a grade-2 meniscal signal does not reach the surface and is not a tear
    ("Grade 2 signal within the posterior horn of the medial meniscus.",
     "Complete tear of the posterior horn of the medial meniscus.", "Medial Meniscus"),
    # a graded ligament lesion outranks intrasubstance degeneration
    ("Mucoid degeneration of the anterior cruciate ligament.",
     "Grade 2 sprain of the anterior cruciate ligament.", "ACL"),
    # chronic subchondral change is weaker evidence of contusion than a stated mechanism
    ("Subchondral cystic change with reactive marrow oedema.",
     "Bone marrow oedema consistent with acute impaction contusion.", "Contusion"),
]


def _selftest_lexicon():
    bad = []
    for text, tgt, want in _LEXICON_CASES:
        v = extract(text)[tgt]
        ok = v > 0.60 if want == "pos" else v < 0.30
        if not ok:
            bad.append(f"{tgt}={v:.2f} want {want} :: {text.splitlines()[0][:52]}")
    for lo_txt, hi_txt, tgt in _LEXICON_ORDER:
        lo, hi = extract(lo_txt)[tgt], extract(hi_txt)[tgt]
        if not lo < hi:
            bad.append(f"{tgt}: {lo:.2f} !< {hi:.2f} :: {lo_txt[:44]}")
    if bad:
        log(f"LEXICON SELF-TEST: {len(bad)} FAILURE(S)")
        for b in bad:
            log("  " + b)
    else:
        n = len(_LEXICON_CASES) + len(_LEXICON_ORDER)
        log(f"lexicon self-test: {n}/{n} pass")
    return not bad


_selftest_lexicon()

In [ ]:
# =============================================================================
# 6. Where the data is, and where the labels come from
# =============================================================================


def find_root():
    """Locate the competition mount without hardcoding one layout.

    The slug appears both singular and plural in different attachments of this dataset, and
    on some accounts the mount is nested one level deeper than usual. A run that dies here
    has spent nothing and scores nothing, so it is worth four probes and a scan.
    """
    cands = [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
             Path("/kaggle/input/rsna-knee-abnormality-detection"),
             Path("/kaggle/input/competitions/rsna-knee-abnormalities-detection"),
             Path("/kaggle/input/rsna-knee-abnormalities-detection"),
             Path("data"), Path(".")]
    for c in cands:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError(
        f"competition mount not found (cwd {Path.cwd()}); expected a directory holding "
        f"test.csv and test_series/")


def find_dinov2(variant="small"):
    """Locate a mounted DINOv2 checkpoint directory by variant name.

    Internet is off in a scored commit, so the encoder arrives as an attached dataset.
    Matching on `config.json` plus `dinov2` in the path finds it wherever it was mounted,
    and the series directories are pruned because walking them costs minutes.

    Local candidates are checked too, so the notebook runs outside Kaggle - against a
    downloaded checkpoint or a synthetic one - without editing this function.
    """
    hits = []
    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "config.json" in files and "dinov2" in root.lower():
                hits.append(Path(root))
    if not hits:
        # Explicit candidates rather than a walk: cwd can be an arbitrary tree, and a
        # recursive scan of it is both slow and unpredictable.
        for p in (list(Path(".").glob("*dinov2*")) + list(Path("weights").glob("*dinov2*"))
                  + list(Path("data").glob("*dinov2*"))):
            if (p / "config.json").is_file():
                hits.append(p)
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None


ROOT = find_root()
log(f"input root: {ROOT}")

LABEL_COLS = TARGETS + [t + "__conf" for t in TARGETS]


class LabelSourceError(RuntimeError):
    """Raised when the labels did not come from where this run intended.

    Every other failure in this file is better survived than reported: a run that dies
    after the cache is built has spent the expensive half and scores nothing, so the guard
    around `main` swallows it and leaves the benchmark file behind. This one is the
    exception. Training on the weaker labels does not look like a failure - it completes,
    writes a plausible submission, and differs only in a log line - so it has to stop the
    run rather than be absorbed by a guard designed for crashes.
    """


def find_label_table():
    """Locate a mounted table of pre-read report labels, if one is attached.

    The lexicon matches morphology, and its failure mode is silence. Silence is measurable
    without any ground truth - for each (report, finding) pair, did anything match? - and
    that measurement says the misses concentrate in particular languages, on findings a
    knee report almost always comments on.

    Enumerating morphology for nine languages is the wrong instrument for that. Reading the
    sentence is the right one, and a language model reads it: against the 58 annotated
    studies the difference is large and one-sided (+0.007 LB). So when such a table is
    mounted it is preferred; when it is not, the lexicon runs and the pipeline is
    unchanged. Both paths produce the same columns, so nothing downstream knows which one
    supplied them.
    """
    base = Path("/kaggle/input")
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            cands += [Path(root) / f for f in files if f.startswith("report_labels")
                      and f.endswith(".csv")]
    cands += [p for p in (Path("data/derived/report_labels_v2.csv"),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if "StudyInstanceUID" in head.columns and all(t in head.columns for t in TARGETS):
            return c
    return None


def label_mount_attached():
    """True when an input directory was attached that is meant to carry a label table.

    The fallback below is deliberate and has to stay silent for a run with no table
    attached, because that is the ordinary case for anyone reading this notebook. It must
    not stay silent for the other case: a table WAS attached and could not be used. Those
    two are indistinguishable from the labels alone - both end with the lexicon - so they
    are separated here by whether the mount exists at all.
    """
    base = Path("/kaggle/input")
    if not base.is_dir():
        return False
    return any("label" in p.name.lower() for p in base.iterdir() if p.is_dir())


def label_coverage(lab):
    """Log, per target, how much of the corpus the labels actually say anything about.

    This is the only quality measurement available on 4,349 unlabelled studies, and it is
    the number every lexicon change in this repository was judged by. A target whose
    silence is high is a target whose labels are mostly the 0.28 prior, and no amount of
    vision model recovers from that.

    Silence is read from the CONFIDENCE, not from a count of matched clauses. `_grade`
    returns confidence 0.05 exactly when nothing matched, and every path that does produce
    evidence returns at least 0.18 - so one threshold covers all twelve targets and all
    scoring paths. Counting matched clauses instead reports the three OA compartments as
    silent on most of the corpus, because `tricompartmental gonarthrosis` names no
    compartment and is inherited rather than matched locally: the gauge would say the
    inheritance rule does nothing, when it is one of the switches worth several points. The
    same applies to the synovitis backoff. Reading confidence also means the gauge works
    unchanged on a mounted label table, which supplies scores and confidences but has no
    clause counts to report.
    """
    log("label coverage (silent / indirect / positive):")
    rows = []
    for t in TARGETS:
        conf = lab[t + "__conf"].to_numpy(dtype=float)
        score = lab[t].to_numpy(dtype=float)
        silent = float(np.mean(conf < 0.10))
        # Evidence that is real but second-hand: an inherited whole-joint statement, a
        # backoff, a hedged clause. Worth separating, because it moves the ranking without
        # asserting the finding.
        indirect = float(np.mean((conf >= 0.10) & (conf < 0.45)))
        pos = float(np.mean(score > 0.5))
        rows.append((t, silent, indirect, pos))
        flag = "  <-- mostly prior" if silent > 0.80 else ""
        log(f"    {t:<17} silent {silent * 100:5.1f}%  indirect {indirect * 100:5.1f}%  "
            f"pos {pos * 100:5.1f}%{flag}")
    return rows


def read_labels(train_df):
    """Labels for every training study, from a mounted table or from the lexicon.

    Studies the mounted table does not cover fall back to the lexicon rather than being
    dropped, so a partial table degrades coverage instead of losing rows.
    """
    n = len(train_df)
    t_lex = time.time()
    lab = pd.DataFrame([extract(r) for r in train_df["Report"].fillna("")])
    lab["StudyInstanceUID"] = train_df["StudyInstanceUID"].values
    lab = lab.set_index("StudyInstanceUID")
    log(f"lexicon read {n} reports in {time.time() - t_lex:.1f}s")

    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError(
                "LABEL SOURCE: a label dataset is mounted but no usable table was found "
                "in it. Falling back to the lexicon here would train on the weaker labels "
                "and say so only in a log line, so the run stops instead.")
        log(f"LABEL SOURCE: lexicon, {n} studies (no table mounted)")
        label_coverage(lab)
        return lab

    tab = pd.read_csv(src).set_index("StudyInstanceUID")
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(
            f"LABEL SOURCE: {src} is missing {len(missing)} expected columns "
            f"(first: {missing[0]!r}). Refusing to fall back silently.")
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(
            f"LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.")
    log(f"LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, "
        f"lexicon for the remaining {n - len(hit)}")
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    label_coverage(lab)
    return lab

## §7 The pixel contract

Everything below this point exists to make one guarantee: **the same anatomy lands on the
same patch tokens regardless of which scanner, protocol or knee produced it.** Four
independent failures of that guarantee were each measurable in this repository, and each
has a rule here.

| what breaks it | what the corpus showed | the rule |
|---|---|---|
| **Scanner field of view** varies 70–320 mm, so a fixed-pixel resize makes 1 px mean anything from 0.2 to 1.4 mm | a 160 mm crop is *larger than the image* in 60% of series and is silently skipped for all of them | crop a fixed **130 mm** box, then resize — mm/px is constant at 0.39 |
| **Slice order on disk** is not anatomical order | filename order correlates with position along the slice normal at **ρ ≈ 0.01** | order by projecting `ImagePositionPatient` onto the slice normal |
| **Left knees are mirror images of right knees** | 4 of 12 targets name `Medial` or `Lateral`; a flip swaps them | canonicalise every study to a right knee **once**, and never flip in augmentation |
| **`ImagePositionPatient` is a corner, not the centre** | corner-x agrees with the `Laterality` tag 58.8% of the time; walking half the extent along the direction cosines to the image centre agrees **96.9%** | derive the side from the image centre, and only where the tag is missing |

Two further details are easy to get wrong and cost real AUC:

**Normalise per series, not per slice.** A knee stack has bright fluid on some slices and
none on others. Percentile-normalising each slice independently rescales those differences
away — it is the inter-slice contrast that says *effusion*. The window is taken over the
whole ordered volume.

**The foreground box recentres the crop; it does not resize it.** Roughly 15–20% of a knee
MRI is air, and it is tempting to crop to the tissue and resize. That restores per-study
scaling and throws away the constant mm/px the resolution argument rests on. So the
tissue bounding box only chooses *where* the fixed 130 mm box sits.

**A sanity check that is cheap and catches everything.** After canonicalisation the medial
compartment is on a fixed side of every sagittal stack. If a change to any rule above
breaks that, the four side-specific targets collapse toward 0.5 while the other eight hold
— which is exactly the signature to look for when the score moves and nothing else explains
it.

In [ ]:
# =============================================================================
# 8. Torch, devices, and how much of this machine the cache may have
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import pydicom

torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


def _cuda_execution_probe(index):
    """Run something real on a device before trusting it.

    `torch.cuda.is_available()` is not the same claim as "this device executes kernels".
    An accelerator that enumerates but fails on first launch turns a working run into a
    crash an hour in, after the cache is built and the budget is spent. A 16x16 convolution
    costs nothing and answers the question that matters.
    """
    dev = torch.device(f"cuda:{index}")
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f"unexpected CUDA probe shape {tuple(out.shape)}")
        torch.cuda.synchronize(index)
        log(f"cuda:{index} probe PASS (compute {major}.{minor}, "
            f"{torch.cuda.get_device_properties(index).total_memory / 1024 ** 3:.0f} GB)")
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        log(f"cuda:{index} probe FAIL ({type(exc).__name__}: {exc}); skipping it")
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False


DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())
            if _cuda_execution_probe(i)]
if not DEVS:
    DEVS = [torch.device("cpu")]
log(f"devices: {[str(d) for d in DEVS]}")

AMP = DEVS[0].type == "cuda"


def available_gb():
    """Memory this machine will actually lend, read rather than assumed.

    A hardcoded ceiling is a guess about a machine the author is not sitting at: too low
    costs coverage silently, too high ends the run. The machine will say, so it is asked.
    """
    try:
        with open("/proc/meminfo") as fh:
            info = {k.strip(): v for k, v in
                    (l.split(":", 1) for l in fh if ":" in l)}
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION


def plan_cache(n_study, n_test=0):
    """Choose how many slices per slot the memory the machine has will allow.

    The cache is n_study x N_SLOT x slices x IMG^2 bytes. Coverage is the cheap axis -
    linear - and resolution the expensive one, so when the budget binds it is the slice
    count that gives way rather than the pixel grid. Deciding once, from the training corpus
    size, keeps the train and test caches on the same group layout, which they must be for
    one set of weights to read both.
    """
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    # Both caches are held at once, and the test half is what a visible run cannot show:
    # here it is a handful of studies, and at scoring it is ~1,300. Sizing against the
    # training corpus alone therefore passes every run that can be watched and overruns the
    # one that counts.
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f"memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; "
        f"sizing for {n_study} train + {n_total - n_study} test studies "
        f"-> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot"
        + (f" (wanted {N_GROUP_MAX})" if groups < N_GROUP_MAX else ""))
    return groups


TRAIN_DF = pd.read_csv(ROOT / "train.csv")
TEST_DF = pd.read_csv(ROOT / "test.csv")
N_GROUP = plan_cache(len(TRAIN_DF), len(TEST_DF))
CACHE_SLICES = GROUP * N_GROUP
log(f"cache layout: {N_GROUP} group(s) x {GROUP} slices = {CACHE_SLICES} per slot; "
    f"train {len(TRAIN_DF)} studies, test {len(TEST_DF)} visible")

# The Report column is the training target and does not exist in the test split. Asserting
# it here rather than discovering it at inference is the difference between a clear message
# and a KeyError forty minutes in.
assert "Report" in TRAIN_DF.columns, "train.csv has no Report column"
if "Report" in TEST_DF.columns:
    log("NOTE: test.csv carries a Report column in this mount; it is ignored by design - "
        "the scored test set does not have one, so nothing may read it.")

In [ ]:
# =============================================================================
# 9. The header pass: what each series is
#
# One header read per series, not per slice. Roughly 20k series against ~10^6 files, so
# this pass is minutes and the slice-ordering pass below is the expensive one.
# =============================================================================
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "ImageLaterality",
            "PixelSpacing", "Rows", "Columns", "RescaleSlope", "RescaleIntercept",
            # Position and orientation come from the same header probe() already opens, so
            # they cost nothing, and they are what recovers the side when the Laterality
            # tag is absent - which it is for a large share of studies here.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Parse a DICOM multi-value string as stored by probe(): floats joined by `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def probe(item):
    """Read one series: its file list and the header of its middle slice.

    The middle slice, not the first. The first slice of a knee stack is often outside the
    joint, and on some protocols its geometry tags differ from the body of the series.
    """
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Every series directory of a split, with one header read per series.

    An absent split returns an empty frame *with the columns annotate expects*. Returning a
    bare DataFrame looks like the same thing and is not: the next call indexes
    `SeriesDescription` and raises KeyError, so the branch that exists to survive a missing
    split is what turns it into a crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    df = pd.DataFrame(rows)
    for c in HDR_TAGS:
        if c not in df.columns:
            df[c] = None
    return df


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header.

    These two axes decide which slot a series fills, and neither is a DICOM field. They are
    reconstructed from the description, from ScanOptions, and - where the description says
    nothing, which is common - from TR and TE directly.
    """
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as exact
    # tokens; a substring test on "SAT" fires on series that are not fat-suppressed.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1 = desc.str.contains(_T1_RX)
    t2 = desc.str.contains(_T2_RX)
    pdw = desc.str.contains(_PD_RX)

    # The description wins where it is explicit; TR/TE decide the rest. A short TR is T1, a
    # long TE is T2, and a long TR with a short TE is proton density. Note that these are
    # mutually exclusive branches evaluated in order - an earlier version tested a T1
    # signature (short TE and short TR) inside the fat-suppression rule as well, which made
    # the two predicates contradict each other and left the coronal T1 slot permanently
    # empty for every study.
    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


_PLANE_AXES = {0: "Sagittal", 1: "Coronal", 2: "Axial"}


def plane_from_iop(iop_str):
    """Plane from the direction cosines, for series the provided table does not name.

    The slice normal is the cross product of the two in-plane direction vectors; whichever
    patient axis it points along most strongly names the plane. This is a fallback: the
    competition ships an Anatomical_Plane column and it is used first.
    """
    v = _hdr_vec(iop_str, 6)
    if v is None:
        return None
    k = np.cross(v[:3], v[3:6])
    n = np.linalg.norm(k)
    if not np.isfinite(n) or n < 1e-6:
        return None
    return _PLANE_AXES[int(np.argmax(np.abs(k / n)))]


def build_plane_map(series_tables, headers):
    """SeriesInstanceUID -> plane, from the provided table with a geometry fallback."""
    pm = {}
    for t in series_tables:
        if t is not None and "Anatomical_Plane" in t.columns:
            pm.update(dict(zip(t["SeriesInstanceUID"], t["Anatomical_Plane"])))
    n_tab = sum(1 for v in pm.values() if isinstance(v, str))
    n_geo = 0
    for h in headers:
        if not len(h):
            continue
        for uid, iop in zip(h["SeriesInstanceUID"], h["ImageOrientationPatient"]):
            if not isinstance(pm.get(uid), str):
                p = plane_from_iop(iop)
                if p is not None:
                    pm[uid] = p
                    n_geo += 1
    log(f"plane: {n_tab} from the provided table, {n_geo} recovered from geometry")
    return pm


def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties break toward the stack with the most slices: a thicker stack samples the joint more
    densely, and the slice sampler benefits from the margin.

    A slot with no series matching its predicate stays EMPTY, and no substitute is admitted
    from a neighbouring predicate. Relaxing the weighting to fill a T1 slot draws from the
    pool SAG_FLUID_NOFS selects from, since that pool is what remains once the weighting is
    dropped: over the training corpus it puts one series in two slots for 2,383 of 4,407
    studies and leaves 56% of the T1 slot holding PD or T2. The presence mask would then
    assert a sequence that was never acquired, and the per-finding softmax would divide its
    attention across two identical slots, giving one acquisition about twice the weight it
    carries in a study that genuinely holds both. The mask exists to say a slot is absent,
    which is what an absent slot is.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs) & (g["fluid"] == fluid)
            cand = g[sel]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out

In [ ]:
# =============================================================================
# 10. Geometry: which knee, which order, which pixels
# =============================================================================


def side_from_geometry(h):
    """Study -> 'L' / 'R' / None, from where the image sits in the patient.

    `Laterality` (0020,0060) is Type 2C and may legitimately be absent; in this corpus it is
    missing on about half the studies, and the vendors it is missing from are whole vendors
    rather than scattered series. A study with no tag is not a left knee, but a
    normalisation that only acts on 'R' treats it as one - so half the corpus goes
    unnormalised and the five side-defined targets see that axis reversed on a large
    minority of it.

    The patient coordinate system fixes this without the tag. DICOM patient coordinates are
    LPS: +x is the patient's left, so the centre of a right knee sits at negative x. The
    CENTRE is used rather than `ImagePositionPatient` itself, because that tag is the corner
    of the image and is offset by half a field of view - enough to change the sign on a knee
    near the midline. Walking half the extent along the direction cosines is what moves
    agreement with the tag from 58.8% to 96.9%.

    The median over a study's series is thresholded, not a single series: probe() reads one
    arbitrary slice per series, which on a sagittal stack can sit anywhere across the joint.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = (ipp[:3]
                 + iop[:3] * ps[1] * float(cols) / 2
                 + iop[3:6] * ps[0] * float(rows) / 2)
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        # Studies whose centre falls near the midline are left unresolved rather than
        # guessed: inside 20 mm the rule is no better than chance.
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Study -> 'L' / 'R' / None: the tag where it exists, geometry where it does not.

    The tag comes first and always wins. Geometry is a fallback for the untagged half, and
    it has to earn that role: agreement with the tag is measured on the studies where both
    exist, and if it falls below the gate the fallback is refused outright rather than
    applied to studies that cannot check it.
    """
    geo = side_from_geometry(h)
    tagged, agree = {}, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if "ImageLaterality" in g.columns:
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        # An empty string is not NaN and is not a side, so the first usable character is
        # what is read rather than the value being truthy.
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        if v:
            tagged[st] = v[0]
            if geo.get(st) is not None:
                agree += geo[st] == v[0]
    both = sum(1 for st in tagged if geo.get(st) is not None)
    rate = agree / both if both else 1.0
    trust = rate >= LAT_MIN_AGREEMENT or both < 50

    d, n_tag, n_geo, n_none = {}, 0, 0, 0
    for st in h["StudyInstanceUID"].unique():
        side = tagged.get(st)
        if side is not None:
            n_tag += 1
        else:
            side = geo.get(st) if trust else None
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} "
        f"unresolved; the two agree on {rate:.1%} of the {both} studies carrying both")
    if not trust:
        log(f"{tag}WARNING: geometry agreement {rate:.1%} is below the "
            f"{LAT_MIN_AGREEMENT:.0%} gate, so the fallback was REFUSED. The untagged "
            f"studies stay unnormalised, which costs the four side-specific targets.")
    elif rate < LAT_WARN_AGREEMENT:
        log(f"{tag}WARNING: geometry agreement {rate:.1%} is below the {LAT_WARN_AGREEMENT:.0%} "
            f"expected from this corpus; the fallback is still used but check the mount.")
    return d


# --------------------------------------------------------------------------- #
# slice order
# --------------------------------------------------------------------------- #
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Series in which at least one sampled slice would not decode. A list rather than a counter
# because appending is atomic under the reader threads, and reported rather than swallowed:
# unreported, a decode failure is indistinguishable from a black knee.
DECODE_FAILED = []
NO_GEOMETRY = []


def order_slices(rec):
    """Return the series' files sorted along the through-plane axis.

    A DICOM file name here is a SOP Instance UID, which is assigned arbitrarily. Sorting by
    it produces an order uncorrelated with anatomy - Spearman between file-name rank and
    physical position is about 0.01, i.e. none. Anything that assumes the file order means
    something is then operating on noise: the three channels of a 2.5D input are three
    unrelated views rather than neighbouring slices, "the middle of the stack" is a random
    subset, and reversing slice order to normalise laterality reverses nothing.

    The physical order is recoverable exactly. Each slice carries its position in patient
    coordinates and its two in-plane axes, so projecting the position onto the slice normal
    gives a signed through-plane coordinate that is monotonic along the stack:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` is the fallback. It usually tracks the projection up to sign, but
    interleaved and multi-echo acquisitions need not number slices in the order they occupy
    in space - and the projection is signed in patient coordinates, which is exactly what
    the laterality convention needs.
    """
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        ds = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None or not np.isfinite(k) for k, _ in keyed):
        # A series with no usable geometry keeps its arbitrary order. That is worse than
        # sorting and better than dropping the series, and it is counted so the run says how
        # often it happened instead of degrading silently.
        NO_GEOMETRY.append(rec.get("SeriesInstanceUID", d))
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


# --------------------------------------------------------------------------- #
# pixels
# --------------------------------------------------------------------------- #
def _fg_centre(vol, h, w):
    """Centre of the tissue bounding box, or the image centre when there is no tissue.

    Roughly 15-20% of a knee MRI is air, and the joint is not always centred in the field of
    view. This moves the fixed crop onto the tissue - it does NOT resize it. Cropping to the
    box and resizing would restore per-study scaling and throw away the constant mm/px that
    the whole resolution argument rests on.
    """
    m = vol.max(axis=0)
    thr = FG_THRESH / 255.0 * (m.max() if m.max() > 0 else 1.0)
    ys, xs = np.nonzero(m > thr)
    if len(ys) < 16:
        return h // 2, w // 2
    cy = int(np.clip((ys.min() + ys.max()) // 2, 0, h - 1))
    cx = int(np.clip((xs.min() + xs.max()) // 2, 0, w - 1))
    return cy, cx


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` physically spread slices from one series, at `out_size` pixels.

    Returns uint8 [n_slice, out, out], normalised per SERIES to its 1st-99th percentile.
    Percentiles rather than min/max because MR intensity has no absolute scale and one bright
    vessel would compress the whole dynamic range. Per series rather than per slice because
    a knee stack has bright fluid on some slices and none on others, and normalising each
    slice alone rescales exactly the inter-slice contrast that says `effusion`.

    Reading is the expensive half of this pipeline, so the caller reads once at the largest
    configuration it needs and derives the smaller ones from the returned buffer.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files = rec.get("ordered") or rec["files"]
    d, px = rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Spread the samples over a central band of the stack: the outermost slices of a knee
    # series are mostly soft tissue outside the joint. The band is a constant rather than a
    # literal because how much of the stack is worth reading depends on how many slices are
    # taken - at three the middle is all that fits, while at twelve the ends are worth
    # having, and a Baker cyst sits at the posteromedial end of a sagittal stack.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = (np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo
           else np.array([n // 2]))
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None                      # no shape is known here; see below
        planes.append(a)

    # A slice that would not decode has no shape of its own, and inventing one is how a
    # single unreadable file erases a whole series: a substitute allocated at the resize
    # target while the decoded slices are still native makes the shape check below take the
    # substitute as the authority and zero the good slices with it, leaving a black slot that
    # the presence mask still reports as acquired.
    #
    # A failure is instead filled from the nearest slice that DID decode - the same
    # convention the sampler already uses when the band holds fewer distinct slices than were
    # asked for - and a series where nothing decodes is reported absent, which the mask can
    # express, rather than black, which it cannot.
    got = [k for k, p in enumerate(planes) if p is not None]
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices of one series can still differ in matrix size - multi-echo and some reformats
    # do - and those are genuinely not stackable.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # Constant physical extent, then resize. PixelSpacing varies 3.4x across the corpus, so
    # without this step one pixel means something different in every study and a 14-pixel
    # patch token covers anywhere from 2.8 mm to 9.5 mm of knee.
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            half = want // 2
            cy, cx = _fg_centre(vol, h, w)
            # Keep the box inside the image and keep its SIZE fixed: clamping the centre
            # rather than truncating the box is what preserves the physical extent.
            cy = int(np.clip(cy, half, h - half))
            cx = int(np.clip(cx, half, w - half))
            vol = vol[:, cy - half:cy + half, cx - half:cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32. These buffers queue between the reader threads and the encoder, and
    # at this size a float32 slot-series is several megabytes. Intensity is already normalised
    # into [0, 1] here, so eight bits cost nothing a bilinear resize has not already cost,
    # and the cache is a quarter the size - which is what decides how many slices fit.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


def normalise_laterality(img, plane, lat):
    """Map every knee onto one canonical side, once, before anything learns from it.

    Coronal and axial views of a left and a right knee are mirror images in plane, so the
    horizontal axis is flipped - `dims=[-1]`, the LAST axis of a [slices, H, W] stack. (The
    vertical axis is not a mirror of anything: flipping it puts the tibia above the femur.)

    Sagittal stacks are not in-plane mirrors of each other at all. What differs is the
    direction the stack runs: medial-to-lateral one way for a left knee and the other way for
    a right one. So the SLICE order is reversed instead - `dims=[0]`.

    This is done once here so that no flip is ever needed in augmentation, and no flip may
    ever be added: four of the twelve targets are named `Medial ...` or `Lateral ...`, and a
    mirror swaps them.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])

In [ ]:
# =============================================================================
# 11. Building the cache
#
# Decoding is the expensive half of this notebook - roughly 10^6 small reads over a network
# mount - and it happens exactly once. Everything downstream reads uint8 out of RAM.
# =============================================================================
def order_all(slots_by_study, tag=""):
    """Sort every selected series along its through-plane axis, in parallel, under a ceiling.

    Only the SELECTED series are ordered. pick_slots has already discarded most of the
    corpus, so ordering everything walked would spend the majority of the pass on series no
    epoch will ever read.

    The ceiling is a property of the mount, not of the work: on a slow mount this pass can
    take an hour for the same series that take ten minutes on a fast one. Passing the budget
    means the remaining series keep their file order - which is uncorrelated with anatomy -
    so the count is logged rather than left to be inferred from the score.
    """
    recs, seen = [], set()
    for chosen in slots_by_study.values():
        for row in chosen.values():
            uid = row["SeriesInstanceUID"]
            if uid not in seen:
                seen.add(uid)
                recs.append(row)
    if not recs:
        return {}, 0

    t_start = time.time()
    out, stats = {}, {"ok": 0, "unordered": 0, "skipped": 0}
    lock = threading.Lock()

    def one(row):
        if time.time() - t_start > ORDER_BUDGET_S:
            with lock:
                stats["skipped"] += 1
            return
        files, ok = order_slices({"files": row["files"], "dir": row["dir"],
                                  "SeriesInstanceUID": row["SeriesInstanceUID"]})
        with lock:
            out[row["SeriesInstanceUID"]] = files
            stats["ok" if ok else "unordered"] += 1

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        list(pool.map(one, recs))

    log(f"{tag}slice order: {stats['ok']} series ordered by geometry, "
        f"{stats['unordered']} had none, {stats['skipped']} left in file order after the "
        f"{ORDER_BUDGET_S / 60:.0f} min ceiling ({time.time() - t_start:.0f}s)")
    if stats["skipped"]:
        log(f"{tag}WARNING: {stats['skipped']} series kept their arbitrary file order; "
            f"their 3-slice groups are not neighbouring slices.")
    return out, stats["skipped"]


def build_cache(studies, slots_by_study, lat, order_map, n_slice, out_size, tag="",
                deadline=None):
    """Decode every study's slots into one uint8 array plus a presence mask.

    Returns
        cache : uint8 [n_study, N_SLOT, n_slice, out, out]
        mask  : bool  [n_study, N_SLOT]        - which slots this study actually has
        index : {StudyInstanceUID: row}

    The mask is not decoration. Roughly a third of the studies are missing at least one slot,
    and a missing slot is not a black one: the head's attention softmax has to be told to
    exclude it, or every study without a T1 acquisition teaches the T1 queries what an empty
    array looks like.

    `deadline` is an absolute wall-clock time, not a duration, because this is called twice -
    once for test and once for train - and two independent durations add up to more than the
    run has. The caller splits one budget between them.

    One allocation up front, sized by plan_cache. Threads fill disjoint rows, so no lock is
    needed on the array itself - only on the counters.
    """
    n = len(studies)
    idx = {st: i for i, st in enumerate(studies)}
    nbytes = n * N_SLOT * n_slice * out_size * out_size
    log(f"{tag}cache: allocating {nbytes / 1024 ** 3:.2f} GB for {n} studies x {N_SLOT} "
        f"slots x {n_slice} slices at {out_size}px")
    cache = np.zeros((n, N_SLOT, n_slice, out_size, out_size), dtype=np.uint8)
    mask = np.zeros((n, N_SLOT), dtype=bool)

    t_start = time.time()
    if deadline is None:
        deadline = T0 + TIME_BUDGET * CACHE_BUDGET_FRAC
    done = {"n": 0, "slots": 0, "over": 0}
    lock = threading.Lock()

    def one(st):
        i = idx[st]
        chosen = slots_by_study.get(st, {})
        side = lat.get(st)
        for s, (name, plane, _fluid, _fs) in enumerate(SLOTS):
            row = chosen.get(name)
            if row is None:
                continue
            if time.time() > deadline:
                with lock:
                    done["over"] += 1
                return
            rec = {"files": row["files"], "dir": row["dir"], "px": row["px"],
                   "SeriesInstanceUID": row["SeriesInstanceUID"],
                   "ordered": order_map.get(row["SeriesInstanceUID"])}
            img = read_slot(rec, n_slice=n_slice, out_size=out_size)
            if img is None:
                continue          # mask stays False: absent, which the head can express
            img = normalise_laterality(img, plane, side)
            cache[i, s] = img.numpy()
            mask[i, s] = True
            with lock:
                done["slots"] += 1
        with lock:
            done["n"] += 1
            if done["n"] % 500 == 0:
                el = time.time() - t_start
                log(f"{tag}  {done['n']}/{n} studies, {done['slots']} slots, "
                    f"{el:.0f}s ({el / max(done['n'], 1):.2f}s per study, "
                    f"eta {el / max(done['n'], 1) * (n - done['n']) / 60:.0f} min)")

    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        list(pool.map(one, studies))

    per_slot = mask.sum(axis=0)
    log(f"{tag}cache built in {(time.time() - t_start) / 60:.1f} min: "
        f"{mask.sum()} slots over {n} studies, "
        f"{(mask.sum(axis=1) == 0).sum()} studies with nothing")
    for (name, _p, _f, _fs), c in zip(SLOTS, per_slot):
        log(f"{tag}  {name:16s} {c:5d} / {n}  ({c / max(n, 1):5.1%})")
    if done["over"]:
        log(f"{tag}WARNING: {done['over']} studies were skipped once the decode deadline "
            f"passed; they are all-absent in the mask and will predict the prior.")
    if DECODE_FAILED:
        log(f"{tag}{len(DECODE_FAILED)} series had at least one slice that would not "
            f"decode (filled from the nearest decoded slice, or reported absent)")
    return cache, mask, idx


class CacheError(RuntimeError):
    """The decode produced too little to train or predict on."""


def decode_grant():
    """The absolute time the two decode passes must be finished by.

    A share of what is *left*, never a share of the whole budget measured from T0. This was
    `T0 + TIME_BUDGET * CACHE_BUDGET_FRAC`, an instant the stages above can walk straight past -
    and did: a 28-minute ordering pass inside a 25-minute budget handed decode a deadline that
    had expired 19 minutes earlier, so every study was skipped in 0.0 min, the mask came out
    all-absent, and the run went on to train for half an hour on zeros and submit twelve
    constant columns.

    Hence the floor. Decode is not the discretionary stage - folds and epochs are, and both
    already stop on the same budget - so when the time is gone it takes its floor anyway and
    the training loop gets whatever is left of nothing.
    """
    left = T0 + TIME_BUDGET - time.time()
    grant = max(left * CACHE_BUDGET_FRAC, CACHE_MIN_S)
    if left < CACHE_MIN_S:
        log(f"WARNING: {left / 60:.0f} min of the {TIME_BUDGET / 3600:.1f} h budget remains and "
            f"decode has not started. Taking {grant / 60:.0f} min for it regardless - a run "
            f"with no cache has nothing to submit - so expect training to be cut short. "
            f"Something above overran: check STAGE 4's ordering time against ORDER_BUDGET_S.")
    return time.time() + grant


def check_cache(mask, tag):
    """Refuse to continue on a cache that cannot support a prediction.

    An all-absent mask is not a degraded run, it is a broken one. Every slot is masked out of
    the attention softmax, so every study returns the head's learned prior, so the twelve
    columns come out constant - scoring exactly the 0.5 benchmark that main() already wrote
    before any of this started. The run completes, writes a valid 13-column file, and reports
    the failure in a mid-log WARNING that arrives half an hour before the summary contradicts
    it. Raising here costs seconds and names the cause; carrying on costs the rest of the
    budget and buries it.
    """
    have = mask.any(axis=1)
    frac = float(have.mean()) if len(have) else 0.0
    if frac < CACHE_MIN_FRAC:
        raise CacheError(
            f"{tag} cache: {int(have.sum())} of {len(have)} studies decoded any slot "
            f"({frac:.1%}), under the {CACHE_MIN_FRAC:.0%} floor. There is nothing to "
            f"{'predict from' if tag == 'test' else 'train on'}: the mask is all-absent, every "
            f"head returns its prior and all twelve columns would be constant. Check the "
            f"decode-deadline warning above, then STAGE 3's slots-per-study and STAGE 4's "
            f"ordering time - a stage that overran its share starves this one.")
    if frac < 0.90:
        log(f"{tag}WARNING: thin cache - {int((~have).sum())} of {len(have)} studies decoded "
            f"nothing and will predict the prior.")

In [ ]:
# =============================================================================
# 12. The model: one encoder, six slots, twelve queries
#
# DINOv2 ViT-S/14 is the encoder. It is a self-supervised model, which is what makes it work
# here: this task has 58 expert labels, so a supervised ImageNet head has nothing to
# fine-tune against, and a frozen ImageNet backbone scores 0.47 - indistinguishable from
# chance on most targets. The same backbone fine-tuned scores 0.866.
# =============================================================================
DINO_PATH = find_dinov2()


class SlotHead(nn.Module):
    """Twelve findings, each attending over the six acquisitions it can be seen in.

    A knee study is a BAG of acquisitions, not an image, and which acquisition answers a
    question depends on the question. An effusion is unmistakable on axial fluid-sensitive
    images and nearly invisible on coronal T1; a medial meniscal tear is the reverse. Mean- or
    max-pooling the bag applies one answer to all twelve, so every finding is read from a
    compromise view.

    So each finding gets its own query and its own softmax over the slots. Three details
    matter:

    * The softmax is MASKED by which slots the study actually has. About a third of studies
      are missing at least one acquisition, and a missing slot is a zero array - attending to
      it teaches the query what an empty buffer looks like.
    * A fixed anatomical prior tilts each query toward the slots that can physically show its
      finding (SLOT_PRIOR_TABLE). It is added to the logits, so at strength 0.55 a favoured
      slot is 1.73x as likely to be read and the head can still learn past it. It is part of
      the architecture and therefore has to be reproduced exactly for a stored state dict to
      mean anything.
    * A study with NO usable slot at all would make the masked softmax divide by zero. Those
      rows are unmasked over their zero features instead, which makes the output a constant
      per target - the learned prior, which is the right answer when there is no evidence.
    """

    def __init__(self, d_in, d_hidden=384, dropout=0.1):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, d_hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.zeros(N_SLOT, d_hidden))
        self.query = nn.Parameter(torch.randn(N_TARGET, d_hidden) * d_hidden ** -0.5)
        self.drop = nn.Dropout(dropout)
        # Per-target readout: one weight vector per finding over its own pooled feature, not
        # a shared Linear. The twelve findings do not share a decision boundary.
        self.out_w = nn.Parameter(torch.randn(N_TARGET, d_hidden) * d_hidden ** -0.5)
        self.out_b = nn.Parameter(torch.zeros(N_TARGET))

        prior = torch.zeros(N_TARGET, N_SLOT)
        for name, slots in SLOT_PRIOR_TABLE.items():
            for s in slots:
                prior[TARGET_IDX[name], s] = SLOT_PRIOR_STRENGTH
        # A buffer, not a parameter: it is fixed, and it must travel with the state dict so
        # that a checkpoint cannot be loaded against a different prior table.
        self.register_buffer("prior", prior)
        self.scale = d_hidden ** -0.5

    def forward(self, feat, mask):
        """feat [B, N_SLOT, d_in] -> logits [B, N_TARGET]; mask [B, N_SLOT] bool."""
        h = self.proj(feat) + self.slot_emb           # [B, S, H]
        h = self.drop(h)
        logits = torch.einsum("bsh,th->bts", h, self.query) * self.scale
        logits = logits + self.prior.unsqueeze(0)

        # Studies with nothing: attend uniformly over their zero features rather than over
        # nothing at all.
        m = mask
        empty = ~m.any(dim=1)
        if empty.any():
            m = m | empty.unsqueeze(1)
        logits = logits.masked_fill(~m.unsqueeze(1), float("-inf"))
        w = logits.softmax(dim=-1)                    # [B, T, S]

        pooled = torch.einsum("bts,bsh->bth", w, h)   # [B, T, H]
        return (pooled * self.out_w.unsqueeze(0)).sum(-1) + self.out_b


class Model(nn.Module):
    """The encoder, applied once per slot, and the head, applied once per study."""

    def __init__(self, backbone, d_model, pool=POOL, unfreeze_last=UNFREEZE_LAST):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(d_model * POOL_PARTS[pool])
        self._set_trainable(unfreeze_last)

    def _set_trainable(self, unfreeze_last):
        """Adapt the top of the encoder; leave the bottom alone.

        With 58 expert labels and ~4.3k noisy ones, fine-tuning 22M parameters end to end
        overfits the noise. The early blocks of a self-supervised ViT hold edge and texture
        filters that transfer as they are; only the last few need to learn that this is a
        knee. The final layernorm is unfrozen with them - it is what the pooled features are
        read through, and freezing it holds them at ImageNet-era statistics.
        """
        for p in self.backbone.parameters():
            p.requires_grad = False
        layers = getattr(getattr(self.backbone, "encoder", None), "layer", None)
        n_open = 0
        if layers is not None and unfreeze_last > 0:
            for blk in layers[-unfreeze_last:]:
                for p in blk.parameters():
                    p.requires_grad = True
                    n_open += p.numel()
        ln = getattr(self.backbone, "layernorm", None)
        if ln is not None:
            for p in ln.parameters():
                p.requires_grad = True
                n_open += p.numel()
        self.n_open = n_open

    def encode(self, x):
        """x [N, GROUP, H, W] -> [N, d_model * parts].

        The three slices of a group become the three input channels. They are neighbouring
        slices in physical order - which is the whole point of the ordering pass - so the
        encoder sees local through-plane context in the channel dimension for free, without
        the cost of a 3D backbone.
        """
        out = self.backbone(pixel_values=x)
        h = out.last_hidden_state                     # [N, 1 + P, D]
        cls, pat = h[:, 0], h[:, 1:]
        parts = [cls, pat.mean(dim=1)]
        if self.pool == "cls_mean_focal":
            # Most of a knee image is not the finding. A patch-mean over 576 tokens dilutes a
            # 1 mm tear across the whole joint; the mean of the most active eighth keeps a
            # focal signal legible while staying a smooth function of the input, which a hard
            # argmax is not. Selection is by token norm - parameter-free, so it is identical
            # across runs and safe to fingerprint.
            k = max(1, pat.shape[1] // 8)
            sel = pat.norm(dim=-1).topk(k, dim=1).indices
            parts.append(pat.gather(1, sel.unsqueeze(-1).expand(-1, -1, pat.shape[-1]))
                         .mean(dim=1))
        return torch.cat(parts, dim=-1)

    def forward(self, bag, mask):
        """bag [B, N_SLOT, GROUP, H, W] (float, ImageNet-normalised) -> [B, N_TARGET]."""
        b, s = bag.shape[:2]
        # Every slot of every study in one encoder call. Looping the slots would launch six
        # small kernels per study and leave the GPU idle between them.
        feat = self.encode(bag.reshape(b * s, *bag.shape[2:])).reshape(b, s, -1)
        return self.head(feat, mask)


def build_model(device):
    """Load the encoder from the attached weights directory and wrap it in the head.

    `AutoModel` rather than `Dinov2Model`, and no `add_pooling_layer` argument: the exact
    transformers version on the image is not under this notebook's control, internet is off,
    and a keyword that a neighbouring release renamed would fail here - after the cache is
    built. The pooling layer this leaves behind is never called; `encode` reads
    `last_hidden_state` directly.
    """
    from transformers import AutoModel
    backbone = AutoModel.from_pretrained(DINO_PATH)
    d = backbone.config.hidden_size
    m = Model(backbone, d).to(device)
    log(f"model: DINOv2 d={d}, pool={POOL} ({POOL_PARTS[POOL]}x{d}={d * POOL_PARTS[POOL]}), "
        f"{m.n_open / 1e6:.1f}M encoder params open of "
        f"{sum(p.numel() for p in backbone.parameters()) / 1e6:.1f}M, "
        f"head {sum(p.numel() for p in m.head.parameters()) / 1e6:.2f}M")
    return m


# ImageNet statistics, because that is what DINOv2 was trained under. The images are
# single-channel MR replicated across a 3-slice group, so all three channels get the same
# statistics - which is correct: they are the same modality, not R, G and B.
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

In [ ]:
# =============================================================================
# 13. Weights packages, and proving a checkpoint is the model you think it is
# =============================================================================
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
OUT_WEIGHTS = WORK / "weights"
MANIFEST_NAME = "manifest.json"
# Folds train concurrently on separate devices and all of them append to one manifest, so the
# read-modify-write below has to be serialised. Without this, two folds that finish together
# each read the same manifest and the second overwrites the first's entry - and a lost member
# only lowers the score, so nothing reports it.
_MANIFEST_LOCK = threading.Lock()


class WeightsError(RuntimeError):
    """An attached package exists but does not describe this model."""


def fingerprint(model, device):
    """Push one fixed synthetic bag through the whole forward pass and keep the output.

    `load_state_dict` returning without complaint proves that the tensor NAMES and SHAPES
    line up. It proves nothing about the arithmetic between them, and the gap is wider than it
    looks. Some divergences are caught by shapes alone - `cls_mean` against `cls_mean_focal`
    changes `proj`'s input width, so that one raises a size mismatch and needs no help. The
    dangerous ones leave every shape intact: the order the parts are concatenated in inside
    `encode`, the fraction of tokens the focal part averages, the `d_hidden ** -0.5` scale on
    the attention logits. Each of those loads perfectly and then predicts noise, and noise on a
    12-class mean AUC looks like 0.5, which looks like a modelling problem rather than a
    loading problem. Days go into that mistake. A fixed input through the full forward catches
    all of them, because the output depends on every step of the graph.

    What it CANNOT catch is worth being precise about, because the obvious reading of the
    check is wrong. Anything the state dict restores is invisible to it by construction: the
    slot prior is a buffer, so loading a package overwrites whatever `SLOT_PRIOR_STRENGTH`
    built and the fingerprint then compares the checkpoint against itself and passes. That is
    the right behaviour - the checkpoint's own prior is the one it was trained under, so it
    should win - but it means a config drift in the prior is corrected silently rather than
    detected, and the notebook's stated constant is no longer what runs.

    Worse, the synthetic bag is `rand`: it has no notion of WHICH slot is which or which
    column is which finding. Permuting `SLOTS` or `TARGETS` therefore produces a bit-identical
    fingerprint while feeding the encoder the wrong acquisition per slot, or writing correct
    predictions under the wrong headers. Neither is detectable from the forward pass at all.
    Those are covered by comparing the manifest instead, in `find_weights` and `load_member` -
    which is the reason the configuration is written next to the tensors.

    The bag is generated from a fixed seed on the CPU, so it is identical on any machine, and
    the check is a tolerance rather than an equality: cuDNN kernel selection and TF32 make the
    last few digits a property of the hardware, not of the model.
    """
    g = torch.Generator().manual_seed(1234)
    bag = torch.rand((1, N_SLOT, GROUP, IMG, IMG), generator=g)
    mask = torch.ones((1, N_SLOT), dtype=torch.bool)
    was_training = model.training
    model.eval()
    with torch.inference_mode():
        out = model(bag.to(device), mask.to(device))
    if was_training:
        model.train()
    return [round(float(v), 6) for v in out.float().cpu().numpy().ravel()]


def check_fingerprint(model, device, expect, name=""):
    """True when this model reproduces a stored fingerprint. Absent -> unverifiable, not bad.

    A third-party checkpoint has no fingerprint from this notebook and cannot be given one
    retroactively. Refusing to load it would discard the checkpoint; loading it silently
    would let a mismatch through unannounced. So it loads, and it is announced.
    """
    if not expect:
        log(f"fingerprint {name}: none stored - loaded UNVERIFIED")
        return None
    got = fingerprint(model, device)
    if len(got) != len(expect):
        raise WeightsError(f"fingerprint {name}: {len(expect)} values stored, "
                           f"{len(got)} produced - this is a different architecture")
    err = max(abs(a - b) for a, b in zip(got, expect))
    ok = err <= FINGERPRINT_TOL
    log(f"fingerprint {name}: max |delta| {err:.2e} vs tol {FINGERPRINT_TOL:.0e} "
        f"-> {'PASS' if ok else 'FAIL'}")
    if not ok:
        raise WeightsError(
            f"fingerprint {name} differs by {err:.3e}. The state dict loaded but the graph it "
            f"loaded into is not the one that produced it. POOL and IMG are the usual causes; "
            f"note that a permuted SLOTS or TARGETS would NOT show up here, so check those "
            f"against the manifest separately.")
    return True


# --------------------------------------------------------------------------- #
# writing a package
# --------------------------------------------------------------------------- #
def save_member(model, device, name, meta):
    """Write one trained member plus everything needed to load it back correctly.

    The point of writing the configuration alongside the tensors is that a later
    inference-only commit attaches this directory and does not have to be trusted to have the
    same constants: it can compare.
    """
    if not SAVE_WEIGHTS:
        return None
    OUT_WEIGHTS.mkdir(parents=True, exist_ok=True)
    path = OUT_WEIGHTS / f"{name}.pt"
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, path)
    entry = dict(meta)
    entry.update({
        "name": name, "file": path.name,
        "fingerprint": fingerprint(model, device),
        "img": IMG, "group": GROUP, "pool": POOL,
        "slots": [s[0] for s in SLOTS],
        "slot_prior_strength": SLOT_PRIOR_STRENGTH,
        "targets": TARGETS,
        "mb": round(path.stat().st_size / 1024 ** 2, 1),
    })
    man_path = OUT_WEIGHTS / MANIFEST_NAME
    with _MANIFEST_LOCK:
        man = {"members": []}
        if man_path.is_file():
            try:
                man = json.loads(man_path.read_text())
            except Exception:
                man = {"members": []}
        man["members"] = [m for m in man.get("members", [])
                          if m.get("name") != name] + [entry]
        man_path.write_text(json.dumps(man, indent=1))
    log(f"saved {name} ({entry['mb']:.0f} MB) -> {path}")
    return entry


# --------------------------------------------------------------------------- #
# reading a package
# --------------------------------------------------------------------------- #
def member_incompatible(m):
    """Which recorded constants of a member disagree with this notebook's, fatally.

    Each of these changes what the tensors MEAN without changing their shape, so the load
    succeeds and the fingerprint - which only re-runs the forward pass the checkpoint already
    defines - cannot see any of them:

    * `img`, `group`   the cache would be a different size than the encoder was adapted to.
    * `pool`           `d_hidden` matches while the feature composition behind it differs.
    * `slots`          `slot_emb` is restored by the load, so the model still expects slot 0
                       to be SAG_FLUID_FS. If `SLOTS` has been reordered, the cache hands it
                       coronal T1 there instead. Every query then reads the wrong acquisition,
                       and the only symptom is a mediocre score.
    * `targets`        the output columns are positional. A reordered `TARGETS` writes correct
                       predictions under the wrong headers, which scores near 0.5 overall
                       while every individual head is fine.

    A member missing a key predates its introduction; those default to compatible rather than
    being discarded, since the alternative is throwing away a checkpoint over a missing
    annotation.
    """
    scalar = [k for k, v in (("img", IMG), ("group", GROUP), ("pool", POOL))
              if m.get(k, v) != v]
    listed = [k for k, v in (("slots", [s[0] for s in SLOTS]), ("targets", TARGETS))
              if list(m.get(k, v)) != list(v)]
    return scalar + listed


def find_weights():
    """Locate an attached package: a directory holding a manifest and its .pt files.

    Returns (dir, members) or (None, []). Nothing here is fatal - a run that finds no package
    trains its own members, which is the whole reason MODE defaults to "auto".
    """
    roots = [OUT_WEIGHTS]
    inp = Path("/kaggle/input")
    if inp.is_dir():
        roots += sorted(p for p in inp.iterdir() if p.is_dir())
    for r in roots:
        for man_path in ([r / MANIFEST_NAME] + sorted(r.glob(f"*/{MANIFEST_NAME}"))):
            if not man_path.is_file():
                continue
            try:
                members = json.loads(man_path.read_text()).get("members", [])
            except Exception as exc:
                log(f"weights: {man_path} is not readable JSON ({exc}); skipping it")
                continue
            usable, base = [], man_path.parent
            for m in members:
                f = base / m.get("file", "")
                if f.is_file():
                    usable.append(m)
                else:
                    log(f"weights: manifest lists {m.get('file')} but it is not on disk")
            if usable:
                log(f"weights: {len(usable)} member(s) in {base}")
                # One pass, so the log and the filter cannot disagree about what was dropped.
                keep = []
                for m in usable:
                    bad = member_incompatible(m)
                    if bad:
                        log(f"  {m.get('name')}: differs on {', '.join(bad)} - it would load "
                            f"and predict nonsense; ignoring it")
                    else:
                        keep.append(m)
                if keep:
                    return base, keep
    log("weights: no attached package found")
    return None, []


def load_member(base, meta, device):
    """Build the model, load one member into it, and verify it before returning it."""
    model = build_model(device)
    sd = torch.load(base / meta["file"], map_location="cpu")
    sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing or unexpected:
        # Reported, not raised: `prior` is a buffer and an older package may not carry it,
        # which is recoverable. The fingerprint is what decides whether the result is usable.
        log(f"load {meta['name']}: {len(missing)} missing, {len(unexpected)} unexpected "
            f"tensor(s){' e.g. ' + missing[0] if missing else ''}")
    # The prior travels in the state dict, so loading has just replaced the one this
    # notebook's constant built. That is correct - the checkpoint was trained under its own
    # prior - but it happens silently, and the fingerprint will pass either way, so the
    # divergence is worth a line rather than a discovery three commits later.
    was = meta.get("slot_prior_strength")
    if was is not None and abs(float(was) - SLOT_PRIOR_STRENGTH) > 1e-9:
        log(f"load {meta['name']}: checkpoint was trained at slot prior {float(was):.3f}, this "
            f"notebook is configured for {SLOT_PRIOR_STRENGTH:.3f}; the checkpoint's value is "
            f"in force (it is a buffer) and the fingerprint below cannot see the difference")
    model.eval()
    check_fingerprint(model, device, meta.get("fingerprint"), meta["name"])
    return model

In [ ]:
# =============================================================================
# 14. Inference: windows, ranks, and writing a file that always exists
# =============================================================================
def window_starts(n_slice, group=GROUP, limit=None):
    """Every start offset of a `group`-slice window inside `n_slice` cached slices.

    Overlapping, not tiling. With 6 cached slices and groups of 3, tiling gives two windows
    (0-2, 3-5) and this gives four (0-2, 1-3, 2-4, 3-5) - a finding straddling the tile
    boundary is in no tiled window and in two of these.

    The off-by-one here is worth stating because it is easy to write and hard to see: the last
    valid start is `n_slice - group`, so the count is `n_slice - group + 1`. Writing
    `range(max(n_slice - group, 0) + 1)[:n_group]` instead - which is the natural shape when
    the cache is thought of as n_group tiles - truncates to the FIRST n_group starts and never
    reads the back of the cache at all. At 12 cached slices and n_group 2 that is windows
    0-2 and 1-3: four slices read, eight decoded and paid for, and every posterior finding
    outside them.
    """
    n = max(n_slice - group + 1, 1)
    starts = list(range(n))
    return starts if limit is None else starts[:limit]


def apply_target_window_pool(p):
    """Pool per-window probabilities into one per target: [W, B, T] -> [B, T].

    Averaging every window is right for a finding that is genuinely everywhere and wrong for
    one that is not. A fracture occupies two or three slices out of twelve; the windows that
    miss it are not weak evidence against it, they are evidence about somewhere else, and
    averaging them in dilutes the one window that saw it by a factor of four. Focal findings
    therefore take the maximum, ligaments - which run through several slices, so a single
    hot window is as likely to be noise as signal - take the mean of the top two, and diffuse
    findings take the mean.
    """
    out = np.empty(p.shape[1:], dtype=np.float64)
    for j, t in enumerate(TARGETS):
        how = TTA_TARGET_POOL.get(t, "mean")
        col = p[:, :, j]
        if how == "max":
            out[:, j] = col.max(axis=0)
        elif how == "top2":
            k = min(2, col.shape[0])
            out[:, j] = np.sort(col, axis=0)[-k:].mean(axis=0)
        else:
            out[:, j] = col.mean(axis=0)
    return out


def _to_input(chunk, device, out_size=None, aug=None):
    """uint8 [B, S, G, H, W] cache slice -> ImageNet-normalised float on the device.

    Geometric augmentation, if any, is applied in [0, 1] space BEFORE normalisation: rotating
    afterwards fills the corners with 0, which in normalised space is roughly -2 standard
    deviations rather than black, and the encoder learns to read that border.
    """
    x = torch.from_numpy(np.ascontiguousarray(chunk)).to(device, non_blocking=True).float()
    x = x.div_(255.0)
    b, s, g = x.shape[:3]
    x = x.reshape(b * s, g, *x.shape[3:])
    if out_size is not None and out_size != x.shape[-1]:
        x = F.interpolate(x, size=(out_size, out_size), mode="bilinear", align_corners=False)
    if aug is not None:
        x = aug(x)
    x = (x - IMNET_MEAN.to(device)) / IMNET_STD.to(device)
    return x.reshape(b, s, g, *x.shape[2:])


@torch.inference_mode()
def predict_member(model, cache, mask, device, rows=None, out_size=None, limit=None, tag=""):
    """Probabilities [n, N_TARGET] from one member, TTA'd over slice windows.

    `rows` selects a subset - the held-out fold, during training - by INDEXING INSIDE the
    batch loop rather than by slicing the cache first. `cache[rows]` would copy the fold out
    of the cache before every epoch's evaluation: at 870 studies, six slots and six slices
    that is a 3.5 GB allocation, ten times per fold, next to the cache it was copied from.
    """
    model.eval()
    n = cache.shape[0] if rows is None else len(rows)
    starts = window_starts(cache.shape[2], GROUP, limit)
    per_window = np.zeros((len(starts), n, N_TARGET), dtype=np.float32)
    for wi, s0 in enumerate(starts):
        for i in range(0, n, EVAL_BATCH):
            sel = slice(i, min(i + EVAL_BATCH, n)) if rows is None \
                else rows[i:i + EVAL_BATCH]
            bag = _to_input(cache[sel][:, :, s0:s0 + GROUP], device, out_size)
            m = torch.from_numpy(mask[sel]).to(device)
            with torch.autocast("cuda", enabled=AMP):
                out = model(bag, m)
            per_window[wi, i:i + (out.shape[0])] = out.float().sigmoid().cpu().numpy()
    log(f"{tag}inferred {n} studies over {len(starts)} window(s) at "
        f"{out_size or IMG}px")
    return apply_target_window_pool(per_window)


def rank_combine(preds, weights=None):
    """Blend members by RANK, per target.

    ROC AUC reads only the ordering of the scores, so nothing outside the ordering is worth
    preserving - and the calibration of a sigmoid trained on soft labels is not comparable
    between members anyway. One member confidently outputting 0.9 and another cautiously
    outputting 0.6 for the same true positive average to something that ranks below a third
    member's overconfident false positive; as ranks they agree. Averaging probabilities lets
    the most poorly calibrated member set the scale of the blend.

    Ranks are normalised to (0, 1) per target so the output still looks like a probability to
    anything downstream, which the submission format expects.
    """
    if not preds:
        return None
    ws = [1.0] * len(preds) if weights is None else list(weights)
    n = preds[0].shape[0]
    acc = np.zeros((n, N_TARGET), dtype=np.float64)
    tot = 0.0
    for p, w in zip(preds, ws):
        if p is None or w <= 0:
            continue
        r = np.empty_like(acc)
        for j in range(N_TARGET):
            # average ranks, so tied scores do not get an arbitrary order
            r[:, j] = pd.Series(p[:, j]).rank(method="average").to_numpy()
        acc += w * r / (n + 1)
        tot += w
    if tot <= 0:
        return None
    return acc / tot


# --------------------------------------------------------------------------- #
# writing the file
# --------------------------------------------------------------------------- #
SUB_PATH = WORK / "submission.csv"


def _write_csv(df, path):
    """Write via a temporary file and replace, so a reader never sees a half-written CSV.

    Also: `index=False`. A pandas index written into a submission adds a nameless first column
    and the grader rejects the file - after the run has spent its nine hours.
    """
    tmp = Path(str(path) + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def write_submission(uids, probs, path=SUB_PATH, tag=""):
    """Write the 12 columns in TARGETS order, with the checks that catch a broken run."""
    df = pd.DataFrame({"StudyInstanceUID": list(uids)})
    for j, t in enumerate(TARGETS):
        df[t] = np.asarray(probs)[:, j]
    assert list(df.columns) == ["StudyInstanceUID"] + TARGETS, "column order"
    assert len(df) == len(uids) and df["StudyInstanceUID"].is_unique, "row set"
    assert df[TARGETS].notna().all().all(), "NaN in the predictions"
    # A constant column scores exactly 0.5 on that target and is nearly always a bug - an
    # untrained head, an all-absent mask, a slot that never filled - not a finding that is
    # genuinely unpredictable.
    flat = [t for t in TARGETS if df[t].nunique() <= 1]
    if flat:
        log(f"{tag}WARNING: constant column(s) {flat} - those targets will score 0.5")
    _write_csv(df, path)
    # Not the column means: after rank_combine every column has mean ~0.5 by construction, so
    # printing means would look identical for a working ensemble and for the 0.5 benchmark.
    # The distinct-value count is what distinguishes them.
    nu = df[TARGETS].nunique()
    log(f"{tag}wrote {path} ({len(df)} rows); {int(nu.min())}-{int(nu.max())} distinct "
        f"values per target out of {len(df)} studies")
    return df


def write_benchmark_submission(uids, path=SUB_PATH):
    """0.5 everywhere, written before anything can fail.

    Nine hours of GPU produce a score only if a file exists at the end. An exception in the
    decode pass, a device that stops responding, a mount that is not shaped the way this
    notebook expects - each of those ends the run, and without this the result is a scoring
    error rather than 0.5. It is overwritten by the first real prediction.
    """
    df = pd.DataFrame({"StudyInstanceUID": list(uids)})
    for t in TARGETS:
        df[t] = 0.5
    _write_csv(df, path)
    log(f"benchmark submission written first ({len(df)} rows, 0.5 everywhere) - "
        f"from here on a crash costs the score, not the submission")
    return df

In [ ]:
# =============================================================================
# 15. Training one member
# =============================================================================
def augment(x):
    """Rigid jitter and a gain change, on a float [N, G, H, W] batch in [0, 1].

    Rotation, scale and translation only. NO FLIP, on either axis, ever:

      * Horizontal flip mirrors the knee, and four of the twelve targets are named
        `Medial ...` or `Lateral ...`. It is not an augmentation of those labels, it is a
        corruption of them - and it is the single most common way to lose AUC on this task,
        because it is the first entry in every default augmentation list and it looks correct.
        The medial/lateral targets fall toward 0.5 while the other eight hold.
      * Vertical flip puts the tibia above the femur, which no knee does.

    The transform is a single affine grid rather than a sequence of resamples: composing the
    three into one matrix costs one interpolation instead of three, and each resample of MR
    data blurs the small high-contrast structures that carry the meniscal and ligament signal.
    """
    n = x.shape[0]
    dev = x.device
    ang = (torch.rand(n, device=dev) * 2 - 1) * (AUG_ROT_DEG * math.pi / 180)
    sc = 1.0 + (torch.rand(n, device=dev) * 2 - 1) * AUG_SCALE
    tx = (torch.rand(n, device=dev) * 2 - 1) * AUG_SHIFT
    ty = (torch.rand(n, device=dev) * 2 - 1) * AUG_SHIFT
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], dim=1),
                         torch.stack([sin, cos, ty], dim=1)], dim=1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    # Gain, not offset: MR intensity has no zero point, so a multiplicative change is the
    # realistic one - it is what a different receive coil or a different normalisation window
    # does. It is applied per image and the result is clamped back into range.
    gain = 1.0 + (torch.rand(n, 1, 1, 1, device=dev) * 2 - 1) * AUG_INTENSITY
    return (x * gain).clamp_(0, 1)


def macro_auc(y_true, y_pred, min_pos=3):
    """Unweighted mean ROC AUC over the targets that have both classes.

    Computed from ranks (the Mann-Whitney identity) rather than through sklearn, because the
    same function has to run on soft targets, on the 58 gold labels, and on whatever subset a
    fold happens to contain, and because a rank AUC has no version-dependent behaviour on
    ties.

    Targets with fewer than `min_pos` positives are SKIPPED rather than scored. A single
    positive gives an AUC that is essentially a coin flip on one study, and averaging that
    into the selection signal makes epoch choice depend on noise. The count of scored targets
    is returned so a caller can see how thin the estimate is.
    """
    aucs = []
    for j in range(y_true.shape[1]):
        y = y_true[:, j]
        pos, neg = y > 0.5, y <= 0.5
        n_p, n_n = int(pos.sum()), int(neg.sum())
        if n_p < min_pos or n_n < min_pos:
            continue
        r = pd.Series(y_pred[:, j]).rank(method="average").to_numpy()
        aucs.append((r[pos].sum() - n_p * (n_p + 1) / 2) / (n_p * n_n))
    return (float(np.mean(aucs)) if aucs else float("nan")), len(aucs)


def assign_folds(df, n_folds=N_FOLDS):
    """Group-stratified folds, grouped on the REPORT TEXT.

    This corpus contains studies whose reports are byte-identical - templated normals, and
    the same patient reported twice. A random split puts those on both sides of the line, and
    since the labels are derived FROM the report, an identical report means an identical label
    vector: the holdout then contains studies whose answer the training set already gave. The
    measured effect is a holdout AUC several points above what the leaderboard returns, which
    is worse than no validation at all, because it is a number that gets trusted.

    Hashing the normalised text groups them. Studies with no report get their own group each -
    they are not duplicates of each other.
    """
    rep = df.get("Report")
    groups = []
    for i, uid in enumerate(df["StudyInstanceUID"]):
        txt = "" if rep is None else str(rep.iloc[i] or "")
        key = normalize(txt)
        groups.append(hashlib.md5(key.encode()).hexdigest() if len(key) > 20 else f"_{uid}")
    order = sorted(set(groups))
    # Assign by a hash of the group rather than by position, so the split does not depend on
    # the order the mount happens to enumerate studies in.
    gf = {g: int(hashlib.md5(f"{SEED}:{g}".encode()).hexdigest(), 16) % n_folds
          for g in order}
    folds = np.array([gf[g] for g in groups])
    log(f"folds: {len(order)} report groups over {len(df)} studies -> "
        + ", ".join(f"f{k}={int((folds == k).sum())}" for k in range(n_folds)))
    return folds


def sample_weights(y, conf, is_gold):
    """Per-study, per-target weight for the BCE.

    Two things are being expressed. First, a label the lexicon was confident about should
    count for more than one it guessed at - but never for nothing, hence the floor: a study
    the lexicon is unsure about is still evidence, and zeroing it would discard most of the
    corpus for the rarer findings. Second, the 58 expert labels are the only ground truth in
    the problem and are worth several noisy ones.
    """
    w = CONF_FLOOR + (1.0 - CONF_FLOOR) * np.clip(conf, 0.0, 1.0)
    w = w * np.where(is_gold[:, None], GOLD_WEIGHT, 1.0)
    return w.astype(np.float32)


def _grad_scaler():
    """A GradScaler that works across the torch versions this image might carry.

    `torch.amp.GradScaler("cuda", ...)` is the current spelling and
    `torch.cuda.amp.GradScaler(...)` the older one. Internet is off and the image version is
    not under this notebook's control, so both are tried rather than assumed.
    """
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=AMP)


def fit_member(name, cache, mask, y, conf, gold, tr_idx, va_idx, device, epochs=EPOCHS,
               deadline=None):
    """Train one fold and return (model, best_auc, history).

    Epoch selection is on the held-out fold's report-derived labels, not on the 58 gold ones.
    That is deliberate and it is the opposite of what it looks like: 58 studies spread over 12
    targets give a handful of positives per target, so a gold-based selection signal moves by
    several points when one study changes rank. The report labels are noisy but there are 4.3k
    of them, and the noise is roughly independent of which epoch is being scored - so they
    rank epochs reliably even though their absolute value is not the competition metric. The
    gold number is computed and printed, and nothing is selected on it.
    """
    model = build_model(device)
    head_p = [p for p in model.head.parameters() if p.requires_grad]
    bb_p = [p for p in model.backbone.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(
        [{"params": head_p, "lr": LR_HEAD},
         {"params": bb_p, "lr": LR_BACKBONE}], weight_decay=WEIGHT_DECAY)

    n_tr = len(tr_idx)
    steps = max(1, math.ceil(n_tr / BATCH_STUDIES)) * epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[LR_HEAD, LR_BACKBONE], total_steps=steps, pct_start=0.15)
    scaler = _grad_scaler()

    w_all = sample_weights(y, conf, gold)
    starts = window_starts(cache.shape[2], GROUP)
    rng = np.random.RandomState(SEED + abs(hash(name)) % 10_000)
    best, best_sd, hist = -1.0, None, []

    for ep in range(epochs):
        model.train()
        perm = rng.permutation(n_tr)
        tot, nb = 0.0, 0
        for b0 in range(0, n_tr, BATCH_STUDIES):
            rows = tr_idx[perm[b0:b0 + BATCH_STUDIES]]
            # A random window per batch is itself an augmentation - across ten epochs a study
            # is seen through every window of its cache, which is the through-plane analogue
            # of a random crop.
            s0 = starts[rng.randint(len(starts))]
            bag = _to_input(cache[rows][:, :, s0:s0 + GROUP], device, aug=augment)
            m = torch.from_numpy(mask[rows]).to(device)
            t = torch.from_numpy(y[rows]).to(device)
            w = torch.from_numpy(w_all[rows]).to(device)
            with torch.autocast("cuda", enabled=AMP):
                out = model(bag, m)
                loss = (F.binary_cross_entropy_with_logits(
                    out.float(), t, reduction="none") * w).sum() / w.sum().clamp(min=1e-6)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for grp in opt.param_groups for p in grp["params"]], 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += float(loss.detach())
            nb += 1

        p_va = predict_member(model, cache, mask, device, rows=va_idx,
                              tag=f"{name} e{ep} ")
        auc, n_scored = macro_auc(y[va_idx], p_va)
        gold_txt = ""
        gi = np.where(gold[va_idx])[0]
        if len(gi) >= 12:
            g_auc, g_n = macro_auc(y[va_idx][gi], p_va[gi])
            gold_txt = f", gold {g_auc:.4f} on {len(gi)} studies / {g_n} targets"
        hist.append((ep, tot / max(nb, 1), auc))
        log(f"{name} epoch {ep}: loss {tot / max(nb, 1):.4f}, holdout {auc:.4f} "
            f"({n_scored}/{N_TARGET} targets scorable){gold_txt}")
        if np.isfinite(auc) and auc > best:
            best = auc
            best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if deadline is not None and time.time() > deadline:
            log(f"{name}: stopping after epoch {ep} - out of time budget")
            break

    if best_sd is not None:
        model.load_state_dict(best_sd)
    model.eval()
    log(f"{name}: best holdout {best:.4f}")
    return model, best, hist

In [ ]:
# =============================================================================
# 16. The run
# =============================================================================
def smoke_subset(h_tr):
    """The train studies a SMOKE run keeps. Returns (filtered headers, kept uids).

    Every gold-labelled study, plus a deterministic sample of the rest up to SMOKE_STUDIES.
    The gold ones are kept deliberately: they are 58 of 4,407, a uniform sample would take one
    or two, and the expert-label branch - the up-weighting, the gold AUC - is exactly the part
    a shakeout should execute. The lexicon still reads every report, so STAGE 5's coverage
    gauge is measured on the whole corpus rather than on this sample.
    """
    gcols = [c for c in TRAIN_DF.columns if c in TARGETS]
    gold_uids = set()
    if len(gcols) == N_TARGET:
        keep = TRAIN_DF[gcols].notna().all(axis=1).to_numpy()
        gold_uids = set(TRAIN_DF.loc[keep, "StudyInstanceUID"])
    rest = [u for u in TRAIN_DF["StudyInstanceUID"] if u not in gold_uids]
    n_take = max(SMOKE_STUDIES - len(gold_uids), 16)
    take = np.random.RandomState(SEED).permutation(len(rest))[:n_take]
    uids = sorted(gold_uids | {rest[i] for i in take})
    log(f"SMOKE: keeping {len(uids)} of {len(TRAIN_DF)} train studies "
        f"({len(gold_uids)} gold + {len(uids) - len(gold_uids)} sampled). Stages 2-4, the "
        f"decode and training see only these; the report lexicon still reads all of them.")
    return h_tr[h_tr["StudyInstanceUID"].isin(set(uids))].reset_index(drop=True), uids


def prepare():
    """Everything between the mount and a pair of caches. Returns one dict."""
    log("=" * 78)
    log("STAGE 1  headers")
    h_tr, h_te = walk("train_series"), walk("test_series")
    log(f"series: {len(h_tr)} train, {len(h_te)} test")
    h_tr, h_te = annotate(h_tr), annotate(h_te)
    smoke_uids = None
    if SMOKE:
        h_tr, smoke_uids = smoke_subset(h_tr)

    def _try(name):
        p = ROOT / name
        try:
            return pd.read_csv(p) if p.is_file() else None
        except Exception:
            return None

    plane_map = build_plane_map([_try("train_series.csv"), _try("test_series.csv")],
                                [h_tr, h_te])

    log("=" * 78)
    log("STAGE 2  which knee")
    lat_tr, lat_te = lat_of(h_tr, "train "), lat_of(h_te, "test ")

    log("=" * 78)
    log("STAGE 3  slots")
    slots_tr = pick_slots(h_tr, plane_map)
    slots_te = pick_slots(h_te, plane_map)
    filled = np.array([len(v) for v in slots_tr.values()])
    log(f"train slots per study: mean {filled.mean():.2f} of {N_SLOT}, "
        f"{int((filled == 0).sum())} studies with none")

    log("=" * 78)
    log("STAGE 4  slice order")
    order_tr, _ = order_all(slots_tr, "train ")
    order_te, _ = order_all(slots_te, "test ")

    log("=" * 78)
    log("STAGE 5  labels")
    lab = read_labels(TRAIN_DF)                 # every report, so the coverage gauge is honest
    uids_tr = smoke_uids if smoke_uids is not None else list(TRAIN_DF["StudyInstanceUID"])
    # One alignment, reused by everything downstream. Row position in TRAIN_DF is not row
    # position in the cache: uids_tr is the subset and the order that y, conf, gold, the folds
    # and the cache rows all share. A boolean mask the length of TRAIN_DF applied to any of
    # them is either an IndexError or - should the lengths ever coincide - silently the wrong
    # studies.
    tdf = TRAIN_DF.set_index("StudyInstanceUID").reindex(uids_tr).reset_index()
    lab = lab.reindex(uids_tr)
    y = lab[TARGETS].to_numpy(dtype=np.float32)
    conf = lab[[t + "__conf" for t in TARGETS]].to_numpy(dtype=np.float32)
    y = np.nan_to_num(y, nan=0.28)      # the corpus-wide positive rate: no information
    conf = np.nan_to_num(conf, nan=0.0)

    # The 58 expert-labelled studies, if the mount exposes them. They are weighted up in the
    # loss and reported separately; nothing is selected on them.
    gold = np.zeros(len(uids_tr), dtype=bool)
    gcols = [c for c in TRAIN_DF.columns if c in TARGETS]
    if len(gcols) == N_TARGET:
        have = tdf[gcols].notna().all(axis=1).to_numpy()
        gold = have
        gy = tdf[TARGETS].to_numpy(dtype=np.float32)
        y[have] = np.nan_to_num(gy[have], nan=0.0)
        conf[have] = 1.0
        log(f"gold labels: {int(gold.sum())} studies carry all {N_TARGET} expert columns; "
            f"they enter the loss at weight {GOLD_WEIGHT}")
    else:
        log("gold labels: train.csv exposes no expert label columns; "
            "training on report-derived labels alone")

    log("=" * 78)
    log("STAGE 6  decode")
    uids_te = list(TEST_DF["StudyInstanceUID"])
    # Test first, and with the smaller share of the budget it needs. A run that decodes all
    # of train and then runs out has nothing to predict with; the reverse merely trains on
    # less. decode_grant works from the time that remains, not from T0 - see its docstring.
    share = len(uids_te) / max(len(uids_te) + len(uids_tr), 1)
    end = decode_grant()
    now = time.time()
    te_deadline = now + (end - now) * min(max(share * 1.5, 0.15), 0.5)
    cache_te, mask_te, _ = build_cache(uids_te, slots_te, lat_te, order_te,
                                       CACHE_SLICES, IMG, "test ", te_deadline)
    check_cache(mask_te, "test ")
    cache_tr, mask_tr, _ = build_cache(uids_tr, slots_tr, lat_tr, order_tr,
                                       CACHE_SLICES, IMG, "train ", end)
    check_cache(mask_tr, "train ")
    del h_tr, h_te, order_tr, order_te
    gc.collect()

    return {"uids_tr": uids_tr, "uids_te": uids_te, "y": y, "conf": conf, "gold": gold,
            "folds": assign_folds(tdf),
            "cache_tr": cache_tr, "mask_tr": mask_tr,
            "cache_te": cache_te, "mask_te": mask_te}


def bank(uids, preds, weights, tag=""):
    """Rank-combine whatever members have finished and overwrite the submission.

    Banking after each member rather than at the end means the file on disk is always the
    best ensemble available so far. If the fourth fold hits an unrecoverable CUDA error, the
    run still submits a three-member ensemble instead of the 0.5 benchmark.
    """
    ok = [(p, w) for p, w in zip(preds, weights) if p is not None and w > 0]
    if not ok:
        return None
    blend = rank_combine([p for p, _ in ok], [w for _, w in ok])
    write_submission(uids, blend, tag=f"{tag}[{len(ok)} member(s)] ")
    return blend


def run_queue(jobs, tag="", on_done=None):
    """Run `jobs` across the available devices, one job per device at a time.

    Two T4s do not make one job twice as fast - a ViT-S at 336 px keeps a single T4 busy - but
    they do halve the wall clock of five independent folds. Each worker owns a device for the
    duration of a job, so no two models share memory on one card. Results go into a
    preallocated list at a fixed index, which is what makes them thread-safe: no shared
    accumulator, no lock around the arithmetic. An earlier version summed ranks into a shared
    array from several threads and lost members to the race, silently, since a lost member
    only lowers the score.

    `on_done` fires as each job finishes, under the lock, with the results so far. It has to
    be here rather than in a loop over the return value: this function BLOCKS until every job
    is done, so anything written afterwards is written once, at the end, which is exactly what
    per-member banking exists to avoid.
    """
    out = [None] * len(jobs)
    nxt = {"i": 0}
    lock = threading.Lock()

    def worker(dev):
        while True:
            with lock:
                i = nxt["i"]
                if i >= len(jobs):
                    return
                nxt["i"] = i + 1
            try:
                res = jobs[i](dev)
            except Exception:
                log(f"{tag}job {i} on {dev} FAILED:\n{traceback.format_exc()}")
                res = None
            finally:
                if dev.type == "cuda":
                    torch.cuda.empty_cache()
            with lock:
                out[i] = res
                if on_done is not None:
                    try:
                        on_done(out)
                    except Exception:
                        log(f"{tag}banking after job {i} failed:\n"
                            f"{traceback.format_exc()}")

    threads = [threading.Thread(target=worker, args=(d,), daemon=True) for d in DEVS]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return out


def train_and_infer(P):
    """Train N_FOLDS members on the report labels and infer the test set from each."""
    folds = P["folds"]                 # assigned in prepare, aligned to P["uids_tr"]
    deadline = T0 + TIME_BUDGET * 0.90

    def make_job(k):
        def job(dev):
            tr = np.where(folds != k)[0]
            va = np.where(folds == k)[0]
            # Relative to the corpus, not absolute. A fold is unusable when it is a
            # degenerate share of what there is - which is what the hash-based grouping can
            # produce if a handful of report groups dominate - and "unusable" means something
            # different for 4,407 studies than for a smoke run over 60.
            min_va = max(3, int(0.02 * len(folds)))
            min_tr = max(3 * BATCH_STUDIES, int(0.20 * len(folds)))
            if len(va) < min_va or len(tr) < min_tr:
                log(f"fold {k}: {len(tr)} train / {len(va)} holdout is below the "
                    f"{min_tr}/{min_va} minimum for a {len(folds)}-study corpus; skipping")
                return None
            name = f"fold{k}"
            model, auc, _hist = fit_member(
                name, P["cache_tr"], P["mask_tr"], P["y"], P["conf"], P["gold"],
                tr, va, dev, deadline=deadline)
            meta = save_member(model, dev, name, {"fold": k, "holdout_auc": auc,
                                                 "n_train": int(len(tr))})
            p_te = predict_member(model, P["cache_te"], P["mask_te"], dev,
                                  tag=f"{name} test ")
            del model
            gc.collect()
            return {"name": name, "auc": auc, "pred": p_te, "meta": meta}
        return job

    def on_done(results):
        # Bank after every completed fold. If fold 4 hits an unrecoverable CUDA error, the
        # file on disk is a three-member ensemble rather than the 0.5 benchmark.
        done = [r for r in results if r]
        bank(P["uids_te"], [r["pred"] for r in done], [1.0] * len(done), "train ")

    results = run_queue([make_job(k) for k in range(N_FOLDS)], "train ", on_done)

    good = [r for r in results if r]
    if not good:
        raise RuntimeError("no fold produced a model - see the tracebacks above")
    log("folds: " + ", ".join(f"{r['name']} {r['auc']:.4f}" for r in good))
    # Equal weights. Weighting folds by their own holdout AUC sounds better and is not: each
    # fold's number comes from ~870 studies and differs between folds by less than its own
    # noise, so the weights fit the validation split rather than the problem.
    return bank(P["uids_te"], [r["pred"] for r in good], [1.0] * len(good), "final ")


def infer_from_package(P, base, members):
    """Predict the test set from an attached package, one member per device."""
    def make_job(m):
        def job(dev):
            model = load_member(base, m, dev)
            p = predict_member(model, P["cache_te"], P["mask_te"], dev,
                               tag=f"{m['name']} ")
            del model
            gc.collect()
            return {"name": m["name"], "pred": p,
                    "w": float(m.get("weight", 1.0))}
        return job

    def on_done(results):
        done = [r for r in results if r]
        bank(P["uids_te"], [r["pred"] for r in done], [r["w"] for r in done], "infer ")

    res = run_queue([make_job(m) for m in members], "infer ", on_done)
    good = [r for r in res if r]
    if not good:
        raise WeightsError("every member of the attached package failed to load or infer")
    log(f"package: {len(good)} of {len(members)} member(s) usable")
    return bank(P["uids_te"], [r["pred"] for r in good], [r["w"] for r in good], "final ")


def main():
    # Before anything that can fail.
    write_benchmark_submission(TEST_DF["StudyInstanceUID"])

    P = prepare()

    base, members = find_weights() if MODE in ("auto", "infer") else (None, [])
    if members:
        log("=" * 78)
        log(f"STAGE 7  inference from {len(members)} attached member(s)")
        infer_from_package(P, base, members)
    elif MODE == "infer":
        raise WeightsError(
            'MODE="infer" was requested but no usable weights package is attached. '
            'Set MODE="auto" to train instead, or attach a package written by a previous '
            'run of this notebook (it saves one to /kaggle/working/weights).')
    else:
        log("=" * 78)
        log(f"STAGE 7  training {N_FOLDS} folds "
            f"({'no package attached' if MODE == 'auto' else 'MODE=train'})")
        train_and_infer(P)

    log("=" * 78)
    log(f"done in {(time.time() - T0) / 60:.1f} min of the "
        f"{TIME_BUDGET / 3600:.1f} h budget")

In [ ]:
# =============================================================================
# 17. Run
# =============================================================================
try:
    main()

except LabelSourceError:
    # Deliberately NOT caught. Every other failure here is better survived than reported,
    # because a crash after the cache is built has spent the expensive half of the run and a
    # 0.5 file is worth more than an error. This one is the exception: training on the weaker
    # labels completes, writes a plausible submission, and differs from the intended run only
    # in a log line. A failure that looks like success has to stop the run.
    raise

except Exception:
    log("=" * 78)
    log("RUN FAILED - the submission on disk is whatever was last banked")
    log(traceback.format_exc())
    if not SUB_PATH.is_file():
        # Failure before the benchmark was written, which means the mount is not shaped the
        # way this notebook expects. Write the file anyway: a scoring error and a 0.5 both
        # score nothing, but only one of them says which stage broke.
        try:
            write_benchmark_submission(pd.read_csv(ROOT / "test.csv")["StudyInstanceUID"])
        except Exception:
            log("could not even read test.csv to write a benchmark file:\n"
                + traceback.format_exc())

finally:
    if SUB_PATH.is_file():
        _sub = pd.read_csv(SUB_PATH)
        log(f"submission.csv: {len(_sub)} rows, {len(_sub.columns)} columns, "
            f"{'0.5 benchmark' if _sub[TARGETS].nunique().max() <= 1 else 'model output'}")
        log(f"total wall clock {(time.time() - T0) / 60:.1f} min")
    else:
        log("NO SUBMISSION FILE - this commit cannot score")

## Appendix

### A. What did not work, and what it cost to find out

Each of these was run. They are recorded because a negative result that is not written down
gets re-run.

| tried | result | why |
|---|---|---|
| Frozen ImageNet backbone + linear head | **0.47** | Chance on most targets. ImageNet features do not separate knee pathology, and with 58 expert labels there is nothing to fine-tune a supervised head against. Self-supervised pretraining is not a refinement here, it is the precondition. |
| Horizontal flip augmentation | −0.03 to −0.06, concentrated in 4 targets | It mirrors medial into lateral. `Medial Meniscus`, `Lateral Meniscus`, `Medial OA` and `Lateral OA` fall toward 0.5 while the other eight hold — the signature to look for whenever those four move together. |
| Sorting slices by filename | ≈ 0 correlation with anatomy | Filenames are SOP Instance UIDs. Spearman against position along the slice normal is **ρ ≈ 0.01**. Every downstream assumption about "neighbouring" slices is then false. |
| 160 mm physical crop | silently inert on 60% of series | Larger than the acquired field of view, so it was skipped for the majority and their scale went unnormalised. Hence 130 mm. |
| `ImagePositionPatient`-x for laterality | 58.8% agreement with the tag | It is the image *corner*. Walking half the extent along the direction cosines to the *centre* gives 96.9%. |
| Per-slice percentile normalisation | worse on Effusion / Synovitis | It rescales away the inter-slice intensity difference that *is* the finding. |
| Filling an empty slot from a neighbouring predicate | one series in two slots for 2,383 / 4,407 studies | 56% of the "T1" slot then held PD or T2, the presence mask asserted an acquisition that was never made, and the per-finding softmax gave one series double weight. An absent slot is better modelled as absent. |
| Weighting folds by their own holdout AUC | no gain, higher variance | Each fold's number comes from ~870 studies and the folds differ by less than that noise. The weights fit the split. |
| Probability averaging instead of rank averaging | −0.004 to −0.011 | AUC reads only the ordering, and sigmoid calibration on soft labels is not comparable between members. The worst-calibrated member sets the scale of a probability blend. |
| Tiled (non-overlapping) TTA windows | −0.006 | A finding on a tile boundary is in no window. Overlapping windows put it in two. |
| Random 5-fold split (not grouped by report) | holdout several points above LB | Byte-identical reports exist in this corpus, and the labels are *derived from* the report — so an identical report is an identical answer sitting on both sides of the split. |
| A smoke run that only shrinks folds and epochs | **0 slots cached, 12 constant columns** | Folds and epochs are not what makes a run long before the first batch. The header pass, the slice ordering and the decode are proportional to the *corpus*: on the real mount they came to 30 minutes, so a 25-minute smoke budget was gone before decode started. `ORDER_BUDGET_S = 3600` inside a 1,500-second run is not a ceiling. Hence `SMOKE_STUDIES`, the clamp on the ordering ceiling, and `check_cache`. |

### A2. Budgets are nested, and the nesting has to be checked

Three budgets bound the run, and they are not independent:

```
TIME_BUDGET                                    the whole run
 ├─ ORDER_BUDGET_S   ≤ TIME_BUDGET × 0.25      stage 4's ceiling
 ├─ decode           ≥ CACHE_MIN_S (20 min)    a share of what REMAINS, floored
 └─ training                                   whatever is left, and it banks per fold
```

Two properties are worth stating because violating either produced a run that reported success
and submitted the benchmark:

- **The ordering ceiling is clamped to a share of the total**, not set absolutely. An absolute
  3,600 s ceiling inside a 1,500 s run lets one stage consume everything downstream while
  truthfully logging `0 left in file order`.
- **The decode deadline is a share of the time that remains, with a floor** — never
  `T0 + TIME_BUDGET × frac`, which is an instant the earlier stages can walk straight past. When
  they do, `build_cache` skips every study in 0.0 min, the mask is all-absent, every head
  returns its prior, and the twelve columns come out constant at exactly the benchmark that was
  already on disk. Decode is not the discretionary stage; folds and epochs are, and both already
  stop on the same clock.

`check_cache` is the backstop: under `CACHE_MIN_FRAC` of studies decoding a single slot it
raises instead of training, because there is nothing to train on and the alternative is
discovering that from a summary line half an hour later.

### B. Validation, honestly

There are **58** expert-labelled studies. Twelve targets over 58 studies is a handful of
positives per target; a per-fold gold AUC moves by points when one study changes rank. So:

- **Epoch and member selection** run on the report-grouped holdout of report-derived labels.
  Noisy in absolute value, but 4.3k studies and the noise is roughly independent of which
  epoch is scored — so it *ranks* epochs reliably, which is all selection needs.
- **The gold-58 number is printed and never selected on.** It is a sanity check, not a
  validation set.
- **Empirically, public LB ≈ gold-58 + 0.044.** Useful for deciding whether a change is worth
  a submission slot; not a substitute for one.

### C. The score ladder, and where this notebook stops

| LB | what it added |
|---|---|
| 0.47 | frozen ImageNet |
| 0.866 | DINOv2-S fine-tuned, 224 px |
| 0.883 | 336 px + physical-scale crop |
| 0.891 | 5-fold ensemble, slot head |
| 0.899 | lexicon audit (directional negation, grade scales, decoys) |
| 0.906 | LLM-read label table where one is attached |
| 0.910 | + cross-series arm — **needs a private checkpoint bundle** |
| 0.920 | + RadImageNet blend — **needs a private checkpoint bundle** |
| 0.922 | + anatomy-query specialist — **needs a private checkpoint bundle** |

**This notebook reproduces the ladder up to and including 0.906**, from the competition mount
plus a public DINOv2-S checkpoint and nothing else. The three arms above it are real and were
measured, but each blends a checkpoint trained in a separate notebook whose weights are not
public — so a rewrite that referenced them would be a rewrite that cannot run. The blend
itself is `rank_combine`, which is already here: attach the members and add them to a
manifest and they enter with no code change.

### D. Attaching a weights package (a two-commit run)

`SAVE_WEIGHTS` writes every trained member plus a manifest to `/kaggle/working/weights`.
To turn that into an inference-only commit:

1. Run once with `MODE="auto"` and no package attached. It trains, saves, and submits.
2. Save the output as a Kaggle dataset.
3. Attach that dataset to a new commit. `find_weights()` locates the manifest, `MODE="auto"`
   sees members and skips training, and the run becomes decode + inference — roughly
   **2 h instead of 5**.

Loading a member is guarded twice, because no single check covers the ways a package can be
wrong.

`member_incompatible` compares the manifest against this notebook's `IMG`, `GROUP`, `POOL`,
`SLOTS` and `TARGETS`. The last two matter most and are the easiest to overlook: `slot_emb` and
the output columns are *positional*, and the load restores them, so a reordered `SLOTS` hands
every query the wrong acquisition and a reordered `TARGETS` writes correct predictions under
the wrong headers. Both run to completion and score near 0.5 while every individual head is
fine.

`check_fingerprint` then re-runs a fixed synthetic bag through the full forward and compares to
the stored output vector at `2e-3`. This covers what the manifest cannot: a graph where every
tensor still fits — the order the parts are concatenated in inside `encode`, the fraction of
tokens the focal part averages, the scale on the attention logits.

Neither check subsumes the other, and it is worth being clear about the seam between them. A
`cls_mean` state dict in a `cls_mean_focal` head is caught by shapes alone, before either.
The slot prior is caught by *neither*: it travels in the state dict as a buffer, so loading
overwrites whatever the notebook configured and the fingerprint then compares the checkpoint
against itself and passes. That is the right outcome — the checkpoint's own prior is the one it
was trained under — but the divergence is silent, so `load_member` logs it explicitly rather
than leaving it to be discovered three commits later.

### E. The efficiency track

Score is `(S − S_bench)/(maxAUC − S_bench) + T/32400`, minimised — so an extra hour is only
worth spending if it buys **+0.044 AUC**. Against that bar:

- **5 folds** (~1 h for ~+0.02) is *not* worth it on the efficiency metric, and is worth it on
  the accuracy metric. It is on by default; set `N_FOLDS = 1` for an efficiency submission.
- **336 px over 224 px** (+0.017 for ~35 min) is marginal either way; `IMG_EFFICIENCY = 224`
  exists for that variant. Note it changes the fingerprint, so a 224 px run needs its own
  package.
- **Overlapping-window TTA** costs minutes and is worth it on both.

A reasonable efficiency configuration: `N_FOLDS = 1`, `IMG = IMG_EFFICIENCY`,
`N_GROUP_MAX = 1`, everything else unchanged — roughly 2 h end to end.

### F. Reading order for the reproducer

The parts most likely to be wrong in a re-implementation, in order of how much they cost:

1. `normalise_laterality` — flipping the wrong axis is invisible in a log and fatal to four
   targets. A `[K, H, W]` stack's horizontal axis is `-1`, not `0`.
2. `window_starts` — the last valid start is `n_slice − group`, so the count is
   `n_slice − group + 1`. Truncating to the first `n_group` starts decodes twelve slices and
   reads four.
3. `annotate` — the fat-suppression and weighting predicates must not overlap, or a slot
   becomes unfillable for every study and the presence mask hides it.
4. `pick_slots` — no substitutes.
5. `_negated` — negation is directional and language-specific, and `Fractures : Aucune.`
   needs the colon rule or it reads as a positive fracture.